# Molecular Docking Studies II
### PHS 312 - Drug Discovery, Design, and Development
**Lecture 26 · Monday, November 2, 2026 · 50 minutes**

Pre-class reading: Patrick 7e, Sections 17.12.7–17.14

**This activity takes 30-35 minutes.** Work in pairs. Run each cell with the
play button on its left, answer the question that follows, and keep going.
You will not write any code.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Spinner101/phs312-cadd-notebooks/blob/main/notebooks/PHS312_L26_Molecular_Docking_II.ipynb)

---
### Before you start - what this notebook does with your work

* **Nothing you type is sent anywhere.** This notebook makes no network
  calls, writes no files, and does not connect to Google Drive. Your answers
  live in the memory of this browser tab and disappear when you close it.
* **Your instructor cannot see this page.** At the end you will get a short
  report code. You choose to paste it into Canvas - that is the only thing
  that reaches your instructor.
* **Do not type anything personal.** No patient information, no identifiers,
  nothing about a real person. Everything here is simulated.
* Opening this notebook signs you in with your Hampton Google account, which
  is covered by the university's agreement with Google. If you would rather
  not use a Google account, tell your instructor and you will be given an
  offline copy.
---

In [ ]:
#@title Set-up  -  run this cell first { display-mode: "form" }
#@markdown Press the play button once. Nothing is downloaded and nothing
#@markdown leaves this browser tab. Takes a few seconds.
import sys, types, textwrap

_tk = types.ModuleType("toolkit"); _tk.__dict__["__name__"] = "toolkit"
exec(compile("\"\"\"PHS 312 activity toolkit - assessment engine and simulation helpers.\n\nEverything runs in the browser session. Nothing is transmitted anywhere:\nno network calls, no file writes, no analytics. Scores exist only in the\nstudent's own Colab session until they choose to report them in Canvas.\n\"\"\"\nimport textwrap, hashlib, json\nfrom datetime import datetime\n\nW = 86\ndef _wrap(s, indent=\"\"):\n    out = []\n    for para in str(s).split(\"\\n\"):\n        out.extend(textwrap.wrap(para, W, initial_indent=indent,\n                                 subsequent_indent=indent) or [\"\"])\n    return \"\\n\".join(out)\n\ndef _rule(ch=\"-\"):\n    return ch * W\n\n\nclass Item:\n    \"\"\"One multiple-choice item with per-distractor feedback.\"\"\"\n    def __init__(self, qid, stem, options, key, why, bloom=\"\", silo=\"\",\n                 section=\"concept\"):\n        if key not in options:\n            raise ValueError(f\"{qid}: key {key!r} not among options {list(options)}\")\n        missing = [k for k in options if k != key and k not in why]\n        if missing:\n            raise ValueError(f\"{qid}: no feedback written for distractor(s) {missing}\")\n        self.qid, self.stem, self.options = qid, stem, options\n        self.key, self.why = key, why\n        self.bloom, self.silo, self.section = bloom, silo, section\n\n\nclass Gradebook:\n    \"\"\"Tracks responses and reports performance. Local to this session.\"\"\"\n\n    SECTIONS = (\"precheck\", \"concept\", \"post\")\n    LABELS = {\"precheck\": \"Prior-knowledge check\",\n              \"concept\": \"Concept checks during the activity\",\n              \"post\": \"End-of-activity assessment\"}\n\n    def __init__(self, lecture, title):\n        self.lecture, self.title = lecture, title\n        self.items, self.responses = {}, {}\n        self.started = datetime.now()\n\n    # ---------- authoring ----------\n    def add(self, item):\n        if item.qid in self.items:\n            raise ValueError(f\"duplicate qid {item.qid}\")\n        self.items[item.qid] = item\n\n    def add_many(self, items):\n        for it in items:\n            self.add(it)\n\n    # ---------- student interaction ----------\n    def ask(self, qid):\n        \"\"\"Print the stem and options for an item.\"\"\"\n        it = self.items[qid]\n        print(_rule(\"=\"))\n        print(_wrap(f\"{qid}.  {it.stem}\"))\n        print()\n        for letter, text in it.options.items():\n            print(_wrap(f\"{letter}) {text}\", indent=\"    \"))\n        print(_rule(\"=\"))\n\n    def answer(self, qid, choice):\n        \"\"\"Record a response and give feedback appropriate to the section.\"\"\"\n        if qid not in self.items:\n            print(f\"[!] Unknown question id {qid!r}. Check the notebook.\")\n            return\n        it = self.items[qid]\n        choice = (choice or \"\").strip().upper()\n        if choice in (\"\", \"-\", \"SELECT\", \"NONE\"):\n            print(\"Choose an option from the dropdown, then press the play button again.\")\n            return\n        if choice not in it.options:\n            print(f\"[!] {choice!r} is not one of {list(it.options)}.\")\n            return\n\n        first = qid not in self.responses\n        if first:\n            self.responses[qid] = choice      # only the first response scores\n        correct = choice == it.key\n\n        if it.section == \"precheck\":\n            # No feedback yet: revealing it now would teach the answer\n            # before the activity and destroy the pre/post comparison.\n            n = len([q for q in self.responses\n                     if self.items[q].section == \"precheck\"])\n            total = len([q for q in self.items\n                         if self.items[q].section == \"precheck\"])\n            print(f\"Recorded. ({n} of {total} answered)\")\n            if n == total:\n                print(\"\\nAll set - run the next cell to begin the activity.\")\n            return\n\n        print(_rule())\n        if correct:\n            print(f\"Correct - {choice}.\")\n            print(_wrap(it.why.get(\"_correct\", \"\"), \"\"))\n        else:\n            print(f\"Not quite. You chose {choice}.\")\n            print(_wrap(it.why[choice]))\n            print()\n            print(_wrap(f\"The defensible answer is {it.key}. \"\n                        + it.why.get(\"_correct\", \"\")))\n        if not first:\n            kept = self.responses[qid]\n            print(f\"\\n(Your first response, {kept}, is the one on record - \"\n                  f\"this retry is for your own understanding.)\")\n        print(_rule())\n\n    # ---------- reporting ----------\n    def _score(self, section):\n        qs = [q for q, it in self.items.items() if it.section == section]\n        got = sum(1 for q in qs if self.responses.get(q) == self.items[q].key)\n        return got, len(qs)\n\n    def summary(self, show_pre_post=True):\n        print(_rule(\"=\"))\n        print(f\"  {self.lecture} - {self.title}\")\n        print(f\"  Performance summary\")\n        print(_rule(\"=\"))\n        any_missing = False\n        for sec in self.SECTIONS:\n            got, tot = self._score(sec)\n            if tot == 0:\n                continue\n            answered = len([q for q, it in self.items.items()\n                            if it.section == sec and q in self.responses])\n            if answered < tot:\n                any_missing = True\n            pct = 100 * got / tot if tot else 0\n            bar = \"#\" * int(round(pct / 5)) + \".\" * (20 - int(round(pct / 5)))\n            print(f\"  {self.LABELS[sec]:<38} {got:>2}/{tot:<3} {pct:5.1f}%  [{bar}]\")\n            if answered < tot:\n                print(f\"      ({tot - answered} unanswered)\")\n        print(_rule())\n\n        if show_pre_post:\n            pre_g, pre_t = self._score(\"precheck\")\n            post_g, post_t = self._score(\"post\")\n            if pre_t and post_t:\n                pre = 100 * pre_g / pre_t\n                post = 100 * post_g / post_t\n                print(f\"  Before the activity: {pre:5.1f}%     \"\n                      f\"After the activity: {post:5.1f}%\")\n                if post >= pre:\n                    if pre < 100:\n                        g = (post - pre) / (100 - pre)\n                        print(f\"  Normalised learning gain  g = {g:.2f}\"\n                              f\"   {self._gain_note(g)}\")\n                    else:\n                        print(\"  You came in at ceiling on the prior check.\")\n                else:\n                    print(\"  Your post score is below your pre score - look back at the\")\n                    print(\"  concept checks you missed before you leave.\")\n                print(_rule())\n\n        self._missed()\n        if any_missing:\n            print(\"\\n  Some questions are unanswered. Scroll back and complete them\")\n            print(\"  before copying your report code.\")\n        return self\n\n    @staticmethod\n    def _gain_note(g):\n        if g >= 0.7: return \"(high)\"\n        if g >= 0.3: return \"(medium)\"\n        return \"(low - review before the exam)\"\n\n    def _missed(self):\n        missed = [(q, it) for q, it in self.items.items()\n                  if it.section != \"precheck\"\n                  and q in self.responses and self.responses[q] != it.key]\n        if not missed:\n            print(\"  Nothing missed in the graded sections.\")\n            return\n        print(\"  Worth reviewing before the exam:\")\n        for q, it in missed:\n            tag = f\"[{it.bloom}\" + (f\" | SILO {it.silo}\" if it.silo else \"\") + \"]\"\n            print(_wrap(f\"- {q} {tag}: {it.stem}\", indent=\"    \"))\n\n    def report_code(self, student_name=\"\"):\n        \"\"\"A short code the student pastes into Canvas.\n\n        This is a participation signal, not a secure exam score: it is\n        generated in the browser and a determined student could edit it.\n        Weight it accordingly.\n        \"\"\"\n        tags = {\"precheck\": \"PRE\", \"concept\": \"CON\", \"post\": \"POST\"}\n        parts = []\n        for sec in self.SECTIONS:\n            got, tot = self._score(sec)\n            parts.append(f\"{tags[sec]}{got}/{tot}\" if tot else f\"{tags[sec]}-\")\n        body = f\"{self.lecture}|\" + \",\".join(parts)\n        digest = hashlib.sha256((body + \"|phs312\").encode()).hexdigest()[:4].upper()\n        code = f\"{body}|{digest}\"\n        print(_rule(\"=\"))\n        print(\"  COPY THIS LINE INTO THE CANVAS ASSIGNMENT\")\n        print(_rule(\"=\"))\n        print(f\"  {code}\")\n        print(_rule(\"=\"))\n        print(_wrap(\"Nothing was sent anywhere. This code exists only in your \"\n                    \"browser until you paste it into Canvas yourself.\"))\n        return code\n\n\ndef verify_code(code):\n    \"\"\"Instructor-side check that a pasted code has not been casually edited.\"\"\"\n    try:\n        body, digest = code.strip().rsplit(\"|\", 1)\n        ok = hashlib.sha256((body + \"|phs312\").encode()).hexdigest()[:4].upper() == digest.upper()\n        return ok, body\n    except Exception:\n        return False, \"\"\n", "toolkit.py", "exec"), _tk.__dict__)
sys.modules["toolkit"] = _tk
Item, Gradebook, verify_code = _tk.Item, _tk.Gradebook, _tk.verify_code

_sb = types.ModuleType("swissbio"); _sb.__dict__["__name__"] = "swissbio"
exec(compile("# -*- coding: utf-8 -*-\n\"\"\"Curated bioisosteric replacement data for PHS 312 Lecture 31.\n\nSOURCE\n    SwissBioisostere, https://www.swissbioisostere.ch\n    Molecular Modelling Group, SIB Swiss Institute of Bioinformatics and\n    the University of Lausanne. Underlying data derived from ChEMBL 36.\n\nLICENCE\n    CC-BY 4.0  https://creativecommons.org/licenses/by/4.0/\n    Redistribution and adaptation permitted with attribution.\n\nCHANGES MADE (required by CC-BY)\n    Retrieved 20 October 2026 through the SwissBioisostere web interface.\n    Two single-attachment-point (\"side chain\") fragment queries were run:\n    carboxylic acid [*:1]C(=O)O  and primary amide [*:1]C(=O)N.\n    The full result tables (7,714 and 6,955 replacements) were reduced to the\n    highest-frequency entries shown below. Column names were shortened and\n    values were not otherwise altered. No structures were recomputed.\n\nCITE\n    Cuozzo A., Daina A., Perez M.A.S., Michielin O., Zoete V.\n    SwissBioisostere: molecular replacements for drug discovery guided by\n    updated structural, bioactivity and physico-chemical data accessible\n    through a reshaped Web interface. Nucleic Acids Research, 2021.\n    Wirth M., Zoete V., Michielin O., Sauer W.H.B. SwissBioisostere: a\n    database of molecular replacements for ligand design. Nucleic Acids\n    Research 2013, 41(D1), D1137-D1143. doi:10.1093/nar/gks1059\n\nCOLUMNS\n    smiles  replacement fragment, [*:1] marks the attachment point\n    freq    number of matched molecular pairs in which this replacement occurs\n    rank    rank by frequency within that query\n    better / similar / worse\n            how the replacement changed measured activity across those pairs\n    dlogp, dtpsa, dmw\n            mean change in calculated logP, topological PSA and molecular\n            weight on making the replacement\n\"\"\"\n\nFIELDS = (\"smiles\", \"freq\", \"rank\", \"better\", \"similar\", \"worse\",\n          \"dlogp\", \"dtpsa\", \"dmw\")\n\n# common names for the fragments students meet in Patrick 7e 13.3.7\nNAMES = {\n    \"COC(=O)[*:1]\": \"methyl ester\",\n    \"NC(=O)[*:1]\": \"primary amide\",\n    \"CCOC(=O)[*:1]\": \"ethyl ester\",\n    \"O[*:1]\": \"hydroxyl\",\n    \"C[*:1]\": \"methyl\",\n    \"OC[*:1]\": \"hydroxymethyl\",\n    \"O=C(O)C[*:1]\": \"acetic acid (homologated)\",\n    \"N#C[*:1]\": \"nitrile\",\n    \"CNC(=O)[*:1]\": \"N-methyl amide\",\n    \"CO[*:1]\": \"methoxy\",\n    \"N[*:1]\": \"amine\",\n    \"CN(C)C(=O)[*:1]\": \"N,N-dimethyl amide\",\n    \"O=C(NO)[*:1]\": \"hydroxamic acid\",\n    \"c1ccc([*:1])cc1\": \"phenyl\",\n    \"F[*:1]\": \"fluorine\",\n    \"Cl[*:1]\": \"chlorine\",\n    \"n1n[nH]c([*:1])n1\": \"tetrazol-5-yl\",\n    \"FC(F)(F)[*:1]\": \"trifluoromethyl\",\n    \"n1nc([*:1])n[nH]1\": \"tetrazol-5-yl (tautomer)\",\n    \"O=[N+]([O-])[*:1]\": \"nitro\",\n    \"NS(=O)(=O)[*:1]\": \"sulfonamide\",\n    \"CS(=O)(=O)NC(=O)[*:1]\": \"acylsulfonamide\",\n    \"O=C(O)CC[*:1]\": \"propanoic acid (homologated)\",\n    \"O=C(NCCO)[*:1]\": \"N-(2-hydroxyethyl) amide\",\n    \"CC(=O)N[*:1]\": \"acetamido\",\n    \"Br[*:1]\": \"bromine\",\n    \"CS(=O)(=O)[*:1]\": \"methylsulfonyl\",\n    \"O=C(N1CCOCC1)[*:1]\": \"morpholine amide\",\n    \"CC(=O)[*:1]\": \"acetyl\",\n    \"CN(C)[*:1]\": \"dimethylamino\",\n    \"O=C(O)[*:1]\": \"carboxylic acid\",\n    \"CCNC(=O)[*:1]\": \"N-ethyl amide\",\n    \"O=C(NC1CC1)[*:1]\": \"N-cyclopropyl amide\",\n    \"CS(=O)(=O)N[*:1]\": \"methanesulfonamido\",\n}\n\n# ---- carboxylic acid: what replaces [*:1]C(=O)O  (7,714 replacements total)\nACID_TOTAL = 7714\nACID = [\n (\"COC(=O)[*:1]\", 4530, 1, 1238, 2353, 939, 0.09, -11.00, 14.03),\n (\"NC(=O)[*:1]\", 2781, 2, 1000, 1334, 447, -0.60, 5.79, -0.99),\n (\"CCOC(=O)[*:1]\", 2449, 3, 703, 1024, 722, 0.48, -11.00, 28.06),\n (\"O[*:1]\", 1626, 4, 604, 793, 229, 0.01, -17.07, -28.01),\n (\"C[*:1]\", 1478, 5, 620, 616, 242, 0.61, -37.30, -29.98),\n (\"OC[*:1]\", 1144, 6, 395, 546, 203, -0.20, -17.07, -13.98),\n (\"O=C(O)C[*:1]\", 956, 7, 253, 530, 173, -0.08, 0.00, 14.02),\n (\"N#C[*:1]\", 877, 8, 346, 353, 178, 0.17, -13.51, -19.00),\n (\"CNC(=O)[*:1]\", 818, 9, 298, 412, 108, -0.34, -8.20, 13.04),\n (\"CO[*:1]\", 788, 10, 331, 334, 123, 0.31, -28.07, -13.98),\n (\"N[*:1]\", 721, 11, 201, 368, 152, -0.51, -11.28, -29.00),\n (\"CN(C)C(=O)[*:1]\", 692, 12, 230, 331, 131, 0.00, -16.99, 27.07),\n (\"O=C(NO)[*:1]\", 654, 13, 440, 157, 57, -0.58, 12.03, 15.02),\n (\"c1ccc([*:1])cc1\", 635, 14, 271, 262, 102, 1.96, -37.30, 32.09),\n (\"F[*:1]\", 625, 15, 265, 259, 101, 0.44, -37.30, -26.02),\n (\"Cl[*:1]\", 604, 16, 276, 240, 88, 0.95, -37.30, -9.57),\n (\"n1n[nH]c([*:1])n1\", 545, 17, 105, 325, 115, -0.52, 17.16, 24.03),\n (\"FC(F)(F)[*:1]\", 503, 18, 214, 206, 83, 1.32, -37.30, 23.98),\n (\"n1nc([*:1])n[nH]1\", 484, 19, 135, 256, 93, -0.51, 17.16, 24.03),\n (\"O=[N+]([O-])[*:1]\", 384, 20, 166, 167, 51, 0.21, 5.84, 0.99),\n (\"NS(=O)(=O)[*:1]\", 382, 21, 157, 168, 57, -1.05, 22.86, 35.07),\n (\"CS(=O)(=O)NC(=O)[*:1]\", 369, 22, 36, 226, 107, -1.01, 25.94, 77.11),\n (\"O=C(O)CC[*:1]\", 348, 23, 112, 191, 45, 0.32, 0.00, 28.06),\n (\"O=C(NCCO)[*:1]\", 341, 24, 148, 142, 51, -0.98, 12.03, 43.07),\n (\"CC(=O)N[*:1]\", 331, 25, 111, 160, 60, 0.26, -8.20, 13.04),\n (\"Br[*:1]\", 330, 26, 153, 125, 52, 1.06, -37.30, 34.89),\n (\"CS(=O)(=O)[*:1]\", 327, 27, 132, 141, 54, -0.43, -3.16, 34.08),\n (\"O=C(N1CCOCC1)[*:1]\", 315, 28, 93, 151, 71, -0.23, -7.76, 69.10),\n (\"CC(=O)[*:1]\", 296, 29, 120, 118, 58, 0.50, -20.23, -1.97),\n (\"CN(C)[*:1]\", 283, 30, 93, 116, 74, 0.08, -34.06, -0.94),\n]\n\n# ---- primary amide: what replaces [*:1]C(=O)N  (6,955 replacements total)\nAMIDE_TOTAL = 6955\nAMIDE = [\n (\"O=C(O)[*:1]\", 2781, 1, 447, 1334, 1000, 0.60, -5.79, 0.99),\n (\"N#C[*:1]\", 2384, 2, 586, 1189, 609, 1.04, -19.30, -18.01),\n (\"CNC(=O)[*:1]\", 2202, 3, 192, 1591, 419, 0.26, -13.99, 14.03),\n (\"O[*:1]\", 1622, 4, 398, 1003, 221, 0.11, -22.86, -27.02),\n (\"COC(=O)[*:1]\", 1269, 5, 259, 699, 311, 0.68, -16.79, 15.02),\n (\"C[*:1]\", 1238, 6, 361, 668, 209, 1.21, -43.09, -28.99),\n (\"CN(C)C(=O)[*:1]\", 1112, 7, 128, 590, 394, 0.61, -22.78, 28.06),\n (\"OC[*:1]\", 1060, 8, 233, 651, 176, 0.51, -22.86, -13.00),\n (\"CO[*:1]\", 1003, 9, 235, 554, 214, 0.91, -33.86, -12.99),\n (\"F[*:1]\", 720, 10, 195, 360, 165, 1.04, -43.09, -25.04),\n (\"Cl[*:1]\", 669, 11, 262, 281, 126, 1.55, -43.09, -8.58),\n (\"N[*:1]\", 646, 12, 151, 332, 163, 0.08, -17.07, -28.01),\n (\"FC(F)(F)[*:1]\", 635, 13, 194, 272, 169, 1.92, -43.09, 24.97),\n (\"CCOC(=O)[*:1]\", 612, 14, 146, 344, 122, 1.08, -16.79, 29.03),\n (\"CS(=O)(=O)[*:1]\", 599, 15, 90, 403, 106, 0.17, -8.95, 35.06),\n (\"CC(=O)N[*:1]\", 546, 16, 80, 349, 117, 0.26, -13.99, 14.03),\n (\"O=C(NCCO)[*:1]\", 532, 17, 42, 385, 105, -0.38, 6.24, 44.05),\n (\"c1ccc([*:1])cc1\", 476, 18, 128, 246, 102, 2.37, -43.09, 33.07),\n (\"O=C(N1CCOCC1)[*:1]\", 443, 19, 45, 250, 148, 0.37, -13.55, 70.09),\n (\"CC(=O)[*:1]\", 441, 20, 112, 235, 94, 1.10, -26.02, -0.98),\n (\"CCNC(=O)[*:1]\", 438, 21, 65, 269, 104, 0.65, -13.99, 28.06),\n (\"O=C(NC1CC1)[*:1]\", 431, 22, 94, 246, 91, 0.80, -13.99, 40.06),\n (\"NS(=O)(=O)[*:1]\", 382, 23, 72, 232, 78, -0.45, 17.07, 36.06),\n (\"CS(=O)(=O)N[*:1]\", 359, 24, 58, 225, 76, 0.27, 3.08, 50.08),\n (\"Br[*:1]\", 339, 25, 137, 146, 56, 1.66, -43.09, 35.88),\n]\n\nQUERIES = {\n    \"carboxylic acid\": dict(query=\"[*:1]C(=O)O\", total=ACID_TOTAL, rows=ACID),\n    \"primary amide\":   dict(query=\"[*:1]C(=O)N\", total=AMIDE_TOTAL, rows=AMIDE),\n}\n\n\ndef table(which=\"carboxylic acid\"):\n    q = QUERIES[which]\n    return [dict(zip(FIELDS, r), name=NAMES.get(r[0], r[0])) for r in q[\"rows\"]]\n\n\ndef success_rate(row):\n    \"\"\"Fraction of matched pairs where activity did not get worse.\"\"\"\n    n = row[\"better\"] + row[\"similar\"] + row[\"worse\"]\n    return (row[\"better\"] + row[\"similar\"]) / n if n else 0.0\n", "swissbio.py", "exec"), _sb.__dict__)
sys.modules["swissbio"] = _sb
swissbio = _sb

_cr = types.ModuleType("crystal"); _cr.__dict__["__name__"] = "crystal"
exec(compile("# -*- coding: utf-8 -*-\n\"\"\"PHS 312 \u2014 real-structure activities built on crystallographic coordinates.\n\nEverything here operates on a PDB entry embedded in the notebook itself.\nNo network, no install beyond NumPy and matplotlib. Optional 3D viewing via\npy3Dmol when it is available; otherwise matplotlib draws the structure.\n\nStructure: PDB 1ACJ, Harel et al., acetylcholinesterase with tacrine (THA)\nbound in the active-site gorge.\n\"\"\"\nimport base64, gzip, itertools, math\nimport numpy as np\nimport matplotlib.pyplot as plt\nfrom mpl_toolkits.mplot3d import Axes3D  # noqa: F401\n\nPRIMARY, SECOND, MUTED, OK = \"#0B6E8F\", \"#C2410C\", \"#6B7280\", \"#15803D\"\nPDB_B64 = \"...\"          # filled in by the notebook builder\n_cache = {}\n\n# covalent radii, enough to infer bonds from coordinates\n_COV = {\"C\": 0.76, \"N\": 0.71, \"O\": 0.66, \"S\": 1.05, \"H\": 0.31, \"P\": 1.07}\nVDW = {\"C\": 1.70, \"N\": 1.55, \"O\": 1.52, \"S\": 1.80, \"H\": 1.20, \"P\": 1.80}\n\n\ndef _fig(w=9.5, h=4.4):\n    f, ax = plt.subplots(figsize=(w, h))\n    ax.grid(alpha=.25); ax.set_axisbelow(True)\n    for s in (\"top\", \"right\"): ax.spines[s].set_visible(False)\n    return f, ax\n\n\ndef pdb_text():\n    if \"txt\" not in _cache:\n        _cache[\"txt\"] = gzip.decompress(base64.b64decode(PDB_B64)).decode()\n    return _cache[\"txt\"]\n\n\ndef _parse():\n    if \"parsed\" in _cache:\n        return _cache[\"parsed\"]\n    prot, lig = [], []\n    for l in pdb_text().splitlines():\n        if l.startswith((\"ATOM\", \"HETATM\")):\n            rec = dict(name=l[12:16].strip(), resn=l[17:20].strip(),\n                       chain=l[21].strip(), resi=l[22:26].strip(),\n                       x=float(l[30:38]), y=float(l[38:46]), z=float(l[46:54]),\n                       elem=(l[76:78].strip() or l[12:16].strip()[0]).upper())\n            (lig if l.startswith(\"HETATM\") else prot).append(rec)\n    _cache[\"parsed\"] = (prot, lig)\n    return prot, lig\n\n\ndef ligand(resn=\"THA\"):\n    _, lig = _parse()\n    if not isinstance(resn, str):\n        raise TypeError(\n            f\"ligand() wants a residue name such as 'THA', got {resn!r}. \"\n            \"If you meant a distance, pass it as cutoff=...\")\n    a = [r for r in lig if r[\"resn\"] == resn.upper()]\n    if not a:\n        have = sorted({r[\"resn\"] for r in lig})\n        raise ValueError(f\"no residue {resn!r} in this structure; available: {have}\")\n    X = np.array([[r[\"x\"], r[\"y\"], r[\"z\"]] for r in a])\n    return a, X\n\n\ndef protein():\n    prot, _ = _parse()\n    X = np.array([[r[\"x\"], r[\"y\"], r[\"z\"]] for r in prot])\n    return prot, X\n\n\ndef bonds_of(atoms, X, tol=0.45):\n    \"\"\"Infer bonds from covalent radii. Returns (i, j, distance).\"\"\"\n    out = []\n    for i, j in itertools.combinations(range(len(X)), 2):\n        d = float(np.linalg.norm(X[i] - X[j]))\n        lim = _COV.get(atoms[i][\"elem\"], 0.77) + _COV.get(atoms[j][\"elem\"], 0.77) + tol\n        if d <= lim:\n            out.append((i, j, d))\n    return out\n\n\ndef angles_of(atoms, X, bonds):\n    adj = {}\n    for i, j, _ in bonds:\n        adj.setdefault(i, []).append(j); adj.setdefault(j, []).append(i)\n    out = []\n    for c, ns in adj.items():\n        for a, b in itertools.combinations(sorted(ns), 2):\n            v1, v2 = X[a] - X[c], X[b] - X[c]\n            cos = float(v1 @ v2 / (np.linalg.norm(v1) * np.linalg.norm(v2)))\n            out.append((a, c, b, math.degrees(math.acos(max(-1, min(1, cos))))))\n    return out\n\n\n# ---------------------------------------------------------------- L22\ndef measure_geometry(term=\"bond lengths\", resn=\"THA\", show=True):\n    \"\"\"Measure what a force field parameterises, from the real crystal structure.\"\"\"\n    atoms, X = ligand(resn)\n    bl = bonds_of(atoms, X)\n    if term == \"bond lengths\":\n        vals = [d for _, _, d in bl]\n        labels = [f\"{atoms[i]['name']}-{atoms[j]['name']}\" for i, j, _ in bl]\n        unit, ideal, title = \"angstrom\", None, \"every bond in the ligand\"\n    else:\n        ang = angles_of(atoms, X, bl)\n        vals = [v for *_, v in ang]\n        labels = [f\"{atoms[a]['name']}-{atoms[c]['name']}-{atoms[b]['name']}\"\n                  for a, c, b, _ in ang]\n        unit, ideal, title = \"degrees\", (109.5, 120.0), \"every valence angle\"\n\n    if show:\n        f, ax = _fig(10.0, 3.8)\n        order = np.argsort(vals)\n        ax.bar(range(len(vals)), [vals[i] for i in order], color=PRIMARY)\n        ax.set_xticks(range(len(vals)))\n        ax.set_xticklabels([labels[i] for i in order], rotation=90, fontsize=7)\n        ax.set_ylabel(unit)\n        ax.set_ylim(min(vals) * 0.97, max(vals) * 1.02)\n        if ideal:\n            for v, lab in zip(ideal, (\"sp3  109.5\", \"sp2  120\")):\n                ax.axhline(v, color=SECOND, ls=\"--\", lw=1.4)\n                ax.text(len(vals) - .5, v, \"  \" + lab, va=\"center\",\n                        color=SECOND, fontsize=9)\n        ax.set_title(f\"{title}, measured in the crystal structure\")\n        plt.tight_layout(); plt.show()\n\n    v = np.array(vals)\n    if not show:\n        out = dict(term=term, n=len(v), lo=round(float(v.min()), 3),\n                   hi=round(float(v.max()), 3), mean=round(float(v.mean()), 3))\n        return out\n    print(f\"{len(v)} measurements from the real structure\")\n    print(f\"  range  {v.min():.3f} to {v.max():.3f} {unit}\")\n    print(f\"  mean   {v.mean():.3f}   spread {v.std():.3f}\")\n    out = dict(term=term, n=len(v), lo=round(float(v.min()), 3),\n               hi=round(float(v.max()), 3), mean=round(float(v.mean()), 3))\n    if term == \"bond lengths\":\n        ar = [d for _, _, d in bl if 1.33 <= d <= 1.43]\n        sp3 = [d for _, _, d in bl if d > 1.48]\n        print(f\"  aromatic-like (1.33-1.43 A): {len(ar)}\")\n        print(f\"  single sp3-like (> 1.48 A) : {len(sp3)}\")\n        print(\"\\n  One force-field parameter cannot cover both. That is why a\")\n        print(\"  force field carries a different constant for each bond type.\")\n        out.update(aromatic=len(ar), sp3=len(sp3))\n    else:\n        n120 = int(((v > 112) & (v < 128)).sum()); n109 = int(((v > 103) & (v < 116)).sum())\n        print(f\"  near 120 deg (sp2): {n120}      near 109.5 deg (sp3): {n109}\")\n        print(\"\\n  The angles cluster around the two hybridisation ideals, which\")\n        print(\"  is exactly what the bending term is fitted to reproduce.\")\n        out.update(near_120=n120, near_109=n109)\n    return out\n\n\ndef contacts(cutoff=4.5, resn=\"THA\", show=True):\n    \"\"\"Real non-bonded contacts between the ligand and the protein.\"\"\"\n    atoms, L = ligand(resn)\n    prot, P = protein()\n    d = np.linalg.norm(L[:, None, :] - P[None, :, :], axis=2)\n    pairs = []\n    for i in range(len(L)):\n        for j in np.where(d[i] <= cutoff)[0]:\n            r = prot[int(j)]\n            s = (VDW.get(atoms[i][\"elem\"], 1.7) + VDW.get(r[\"elem\"], 1.7))\n            pairs.append((float(d[i, int(j)]), atoms[i][\"name\"], r[\"resn\"],\n                          r[\"resi\"], r[\"name\"], s))\n    pairs.sort()\n    if show:\n        gap = np.array([p[0] - p[5] for p in pairs])\n        col = [\"#DC2626\" if g < -0.4 else (OK if g < 0.5 else MUTED) for g in gap]\n        f, ax = _fig(9.4, 4.2)\n        ax.bar(range(len(gap)), gap, color=col, width=1.0)\n        ax.axhline(0, color=\"#1F2937\", lw=1.6)\n        ax.set_xlabel(f\"ligand-protein atom pairs within {cutoff} A, closest first\")\n        ax.set_ylabel(\"distance minus sum of vdW radii (A)\")\n        ax.set_title(\"zero means exactly at van der Waals contact;\\n\"\n                     \"a real clash sits well below zero, and there are none here\",\n                     fontsize=10)\n        nc = int((gap < -0.4).sum()); nt = int(((gap >= -0.4) & (gap < 0.5)).sum())\n        ax.legend(handles=[\n            plt.Line2D([], [], color=\"#DC2626\", lw=6, label=f\"clash ({nc})\"),\n            plt.Line2D([], [], color=OK, lw=6, label=f\"at contact ({nt})\"),\n            plt.Line2D([], [], color=MUTED, lw=6, label=\"further out\")],\n            fontsize=8, loc=\"lower right\")\n        plt.tight_layout(); plt.show()\n    tight = [p for p in pairs if p[0] < p[5]]\n    clash = [p for p in pairs if p[0] < p[5] - 0.4]\n    if show:\n        print(f\"{len(pairs)} ligand-protein atom pairs within {cutoff} A\")\n        print(f\"  {'dist':>6}  {'sum vdW':>8}  contact\")\n        for p in pairs[:10]:\n            mark = \"  at/inside vdW contact\" if p[0] < p[5] else \"\"\n            print(f\"  {p[0]:>6.2f}  {p[5]:>8.2f}  {p[1]:<4} ... \"\n                  f\"{p[2]}{p[3]} {p[4]:<4}{mark}\")\n        print(f\"\\n  at or just inside the vdW sum : {len(tight)}\")\n        print(f\"  genuinely clashing (>0.4 A in): {len(clash)}\")\n        print()\n        if not clash:\n            print(\"  Nothing clashes. That is what a real binding mode looks like:\")\n            print(\"  the ligand sits AT van der Waals contact with the protein,\")\n            print(\"  never inside it. A docking pose that scores well while pushing\")\n            print(\"  atoms closer than this is wrong, however good its number.\")\n    resid = sorted({(p[2], p[3]) for p in pairs})\n    return dict(cutoff=cutoff, pairs=len(pairs), at_or_inside_vdw=len(tight),\n                clashing=len(clash), residues=len(resid),\n                nearest=round(pairs[0][0], 2) if pairs else None)\n\n\ndef dihedral(p0, p1, p2, p3):\n    \"\"\"Standard IUPAC torsion angle in degrees.\"\"\"\n    b0, b1, b2 = p0 - p1, p2 - p1, p3 - p2\n    b1n = b1 / np.linalg.norm(b1)\n    v = b0 - (b0 @ b1n) * b1n\n    w = b2 - (b2 @ b1n) * b1n\n    x = v @ w\n    y = np.cross(b1n, v) @ w\n    return math.degrees(math.atan2(y, x))\n\n\nCHI1 = {\"ARG\": \"CG\", \"ASN\": \"CG\", \"ASP\": \"CG\", \"CYS\": \"SG\", \"GLN\": \"CG\",\n        \"GLU\": \"CG\", \"HIS\": \"CG\", \"ILE\": \"CG1\", \"LEU\": \"CG\", \"LYS\": \"CG\",\n        \"MET\": \"CG\", \"PHE\": \"CG\", \"PRO\": \"CG\", \"SER\": \"OG\", \"THR\": \"OG1\",\n        \"TRP\": \"CG\", \"TYR\": \"CG\", \"VAL\": \"CG1\"}\n\n\ndef sidechain_torsions(near_ligand=8.0, resn=\"THA\"):\n    \"\"\"Real chi1 torsions of residues lining the pocket.\"\"\"\n    prot, P = protein()\n    _, L = ligand(resn)\n    byres = {}\n    for i, r in enumerate(prot):\n        byres.setdefault((r[\"chain\"], r[\"resi\"], r[\"resn\"]), {})[r[\"name\"]] = P[i]\n    out = []\n    for (ch, ri, rn), at in byres.items():\n        g = CHI1.get(rn)\n        if not g or not all(k in at for k in (\"N\", \"CA\", \"CB\", g)):\n            continue\n        d = float(np.min(np.linalg.norm(L - at[\"CA\"], axis=1)))\n        if d > near_ligand:\n            continue\n        chi = dihedral(at[\"N\"], at[\"CA\"], at[\"CB\"], at[g])\n        out.append(dict(resn=rn, resi=ri, chain=ch, chi1=chi, dist=d))\n    out.sort(key=lambda r: r[\"dist\"])\n    return out\n\n\ndef rotamers(near_ligand=8.0, show=True):\n    \"\"\"Real side-chain torsions cluster at the three staggered minima.\"\"\"\n    rows = sidechain_torsions(near_ligand)\n    chi = np.array([r[\"chi1\"] for r in rows])\n    def bucket(a):\n        a = (a + 360) % 360\n        if 0 <= a < 120: return \"gauche+ (~60)\"\n        if 120 <= a < 240: return \"trans (~180)\"\n        return \"gauche- (~300)\"\n    buckets = [bucket(c) for c in chi]\n    if show:\n        f, ax = _fig(9.4, 4.0)\n        ax.hist(((chi + 360) % 360), bins=36, range=(0, 360), color=PRIMARY)\n        for v, lab in ((60, \"gauche+\"), (180, \"trans\"), (300, \"gauche-\")):\n            ax.axvline(v, color=SECOND, ls=\"--\", lw=1.5)\n            ax.text(v, ax.get_ylim()[1] * .92, \"  \" + lab, color=SECOND, fontsize=9)\n        ax.set_xlim(0, 360); ax.set_xticks(range(0, 361, 60))\n        ax.set_xlabel(\"chi-1 torsion angle (degrees)\")\n        ax.set_ylabel(\"side chains\")\n        ax.set_title(f\"{len(rows)} residues within {near_ligand} A of the ligand, \"\n                     \"measured in the crystal\")\n        plt.tight_layout(); plt.show()\n        print(f\"{'residue':<10}{'chi-1':>9}   rotamer                 dist to ligand\")\n        for r in rows[:12]:\n            print(f\"  {r['resn']}{r['resi']:<6}{r['chi1']:>9.1f}   \"\n                  f\"{bucket(r['chi1']):<22}  {r['dist']:.1f} A\")\n        from collections import Counter\n        c = Counter(buckets)\n        print(f\"\\n  {c}\")\n        print(\"\\n  Side chains do not sit at random angles. They occupy three\")\n        print(\"  staggered wells, which is the same local-minimum structure you\")\n        print(\"  see in a torsion profile - measured here in a real protein.\")\n    from collections import Counter\n    c = Counter(buckets)\n    return dict(n=len(rows), counts=dict(c),\n                near_stagger=int(sum(1 for a in chi\n                                     if min(abs(((a + 360) % 360) - v)\n                                            for v in (60, 180, 300)) < 40)))\n\n\ndef search_cost(n_rot=7, step=30, seconds_each=0.05, show=True):\n    \"\"\"Arithmetic, not simulation: how many structures a grid search needs.\"\"\"\n    per_bond = int(round(360 / step))\n    n = per_bond ** n_rot\n    secs = n * seconds_each\n    yrs = secs / 3600 / 24 / 365\n    if show:\n        print(f\"settings per bond        {per_bond}   ({step} degree steps)\")\n        print(f\"rotatable bonds          {n_rot}\")\n        print(f\"conformations to score   {n:,}\")\n        print(f\"at {seconds_each} s each            {secs/3600:,.1f} hours\", end=\"\")\n        print(f\"   =  {yrs:,.0f} years\" if yrs >= 1 else \"\")\n        if yrs >= 1:\n            print(\"\\n  This is why no docking program does a systematic search.\")\n    return dict(conformations=n, per_bond=per_bond, hours=secs / 3600,\n                years=yrs)\n\n\n# ---------------------------------------------------------------- L24\ndef rmsd(A, B):\n    return float(np.sqrt(((A - B) ** 2).sum(axis=1).mean()))\n\n\ndef move_ligand(dx=0.0, dy=0.0, dz=0.0, rot_deg=0.0, axis=\"z\", resn=\"THA\"):\n    \"\"\"Rigid-body move of the real ligand. Returns the moved coordinates.\"\"\"\n    _, X = ligand(resn)\n    c = X.mean(axis=0)\n    t = math.radians(rot_deg)\n    ax = {\"x\": np.array([1., 0, 0]), \"y\": np.array([0, 1., 0]),\n          \"z\": np.array([0, 0, 1.])}[axis]\n    K = np.array([[0, -ax[2], ax[1]], [ax[2], 0, -ax[0]], [-ax[1], ax[0], 0]])\n    R = np.eye(3) + math.sin(t) * K + (1 - math.cos(t)) * (K @ K)\n    return (X - c) @ R.T + c + np.array([dx, dy, dz])\n\n\ndef pose_rmsd(dx=0.0, dy=0.0, dz=0.0, rot_deg=0.0, resn=\"THA\", show=True):\n    \"\"\"Move the real crystal pose and measure the real RMSD you created.\"\"\"\n    atoms, X = ligand(resn)\n    Y = move_ligand(dx, dy, dz, rot_deg, resn=resn)\n    r = rmsd(X, Y)\n    per = np.linalg.norm(X - Y, axis=1)\n    if show:\n        f, (a1, a2) = plt.subplots(1, 2, figsize=(12.4, 4.2))\n        a1.bar(range(len(per)), per, color=PRIMARY)\n        a1.set_xticks(range(len(per)))\n        a1.set_xticklabels([a[\"name\"] for a in atoms], rotation=90, fontsize=7)\n        a1.set_ylabel(\"displacement (A)\")\n        a1.axhline(r, color=SECOND, ls=\"--\", lw=1.6, label=f\"RMSD = {r:.2f} A\")\n        a1.legend(fontsize=9)\n        a1.set_title(\"how far each atom moved\")\n        a2.scatter(X[:, 0], X[:, 1], s=60, color=MUTED, label=\"crystal pose\")\n        a2.scatter(Y[:, 0], Y[:, 1], s=60, color=SECOND, label=\"your pose\")\n        for i in range(len(X)):\n            a2.plot([X[i, 0], Y[i, 0]], [X[i, 1], Y[i, 1]], color=\"#AAB\", lw=.8)\n        a2.set_aspect(\"equal\"); a2.legend(fontsize=9)\n        a2.set_xlabel(\"x (A)\"); a2.set_ylabel(\"y (A)\")\n        a2.set_title(\"looking down z\")\n        for ax_ in (a1, a2):\n            ax_.grid(alpha=.25)\n            for s in (\"top\", \"right\"): ax_.spines[s].set_visible(False)\n        plt.tight_layout(); plt.show()\n    verdict = (\"counts as the correct pose (under 2 A)\" if r < 2.0\n               else \"would be scored as a wrong pose (2 A or more)\")\n    print(f\"RMSD to the crystal pose : {r:.2f} A   -> {verdict}\")\n    print(f\"furthest atom moved      : {per.max():.2f} A\")\n    print(f\"closest atom moved       : {per.min():.2f} A\")\n    return dict(rmsd=round(r, 2), max_move=round(float(per.max()), 2),\n                correct_pose=bool(r < 2.0))\n\n\ndef pharmacophore(resn=\"THA\", show=True):\n    \"\"\"The real 3-point geometry of the bound ligand.\"\"\"\n    atoms, X = ligand(resn)\n    idx = {a[\"name\"]: i for i, a in enumerate(atoms)}\n    ring = [idx[n] for n in (\"C1\", \"C2\", \"C3\", \"C4\", \"C5\", \"C6\") if n in idx]\n    pts = {}\n    if ring: pts[\"aromatic centroid\"] = X[ring].mean(axis=0)\n    for nm, lab in ((\"N15\", \"amine N (donor)\"), (\"N7\", \"ring N (acceptor)\")):\n        if nm in idx: pts[lab] = X[idx[nm]]\n    names = list(pts)\n    print(\"Three-point pharmacophore, measured in the crystal structure:\")\n    out = {}\n    for a, b in itertools.combinations(names, 2):\n        d = float(np.linalg.norm(pts[a] - pts[b]))\n        out[f\"{a} - {b}\"] = round(d, 2)\n        print(f\"   {a:<22} to {b:<22} {d:5.2f} A\")\n    if show and len(names) == 3:\n        f, ax = _fig(6.4, 5.2)\n        P2 = np.array([pts[n] for n in names])\n        ax.scatter(P2[:, 0], P2[:, 1], s=180,\n                   color=[PRIMARY, SECOND, OK][:len(names)], zorder=3)\n        for i, n in enumerate(names):\n            ax.annotate(n, (P2[i, 0], P2[i, 1]), fontsize=9,\n                        xytext=(8, 6), textcoords=\"offset points\")\n        for i, j in itertools.combinations(range(len(names)), 2):\n            ax.plot(P2[[i, j], 0], P2[[i, j], 1], color=MUTED, lw=1.4)\n            m = P2[[i, j]].mean(axis=0)\n            ax.text(m[0], m[1], f\"{np.linalg.norm(P2[i]-P2[j]):.2f} A\",\n                    fontsize=8, color=MUTED)\n        ax.set_aspect(\"equal\"); ax.set_xlabel(\"x (A)\"); ax.set_ylabel(\"y (A)\")\n        ax.margins(0.28)\n        ax.set_title(\"the pharmacophore is this triangle, not the skeleton\",\n                     pad=16)\n        plt.tight_layout(); plt.show()\n    return out\n\n\n# ---------------------------------------------------------------- L25 / L26\ndef score_pose(Y, resn=\"THA\", cutoff=6.0):\n    \"\"\"A deliberately simple score over REAL coordinates.\n\n    contact reward - clash penalty - desolvation-ish term. It is transparent\n    on purpose: every piece is printed, so the point is not that this is a\n    good scoring function but that you can see exactly what it rewards.\n    \"\"\"\n    atoms, _ = ligand(resn)\n    prot, P = protein()\n    d = np.linalg.norm(Y[:, None, :] - P[None, :, :], axis=2)\n    ser = np.array([VDW.get(a[\"elem\"], 1.7) for a in atoms])[:, None]\n    spr = np.array([VDW.get(r[\"elem\"], 1.7) for r in prot])[None, :]\n    s = ser + spr\n    near = d <= cutoff\n    gap = d - s\n    contact = float(np.sum(np.exp(-((gap[near]) ** 2) / 0.8)))\n    overlap = np.clip(-gap, 0, None)\n    clash = float(np.sum(overlap[near] ** 2) * 20)\n    polar_l = np.array([a[\"elem\"] in (\"N\", \"O\") for a in atoms])[:, None]\n    polar_p = np.array([r[\"elem\"] in (\"N\", \"O\") for r in prot])[None, :]\n    hb_mask = near & polar_l & polar_p & (d > 2.4) & (d < 3.5)\n    hbond = float(hb_mask.sum()) * 2.5\n    total = -(contact + hbond) + clash\n    return dict(score=round(total, 2), contact=round(contact, 2),\n                hbond=round(hbond, 2), clash=round(clash, 2),\n                clashing_atoms=int((overlap > 0.4).any(axis=1).sum()))\n\n\ndef dock_pose(dx=0.0, dy=0.0, dz=0.0, rot_deg=0.0, axis=\"z\", resn=\"THA\",\n              show=True):\n    \"\"\"Move the real ligand in the real gorge and score it.\"\"\"\n    atoms, X = ligand(resn)\n    Y = move_ligand(dx, dy, dz, rot_deg, axis, resn)\n    sc = score_pose(Y, resn)\n    r = rmsd(X, Y)\n    if show:\n        prot, P = protein()\n        d = np.linalg.norm(Y[:, None, :] - P[None, :, :], axis=2)\n        nearby = np.where(d.min(axis=0) <= 8.0)[0]\n        f, ax = _fig(7.2, 6.0)\n        ax.scatter(P[nearby, 0], P[nearby, 1], s=14, color=\"#CBD5E1\",\n                   label=\"gorge atoms within 8 A\")\n        ax.scatter(X[:, 0], X[:, 1], s=70, color=MUTED, label=\"crystal pose\")\n        bad = np.zeros(len(Y), bool)\n        ov = np.clip((np.array([VDW.get(a[\"elem\"],1.7) for a in atoms])[:,None]\n                      + np.array([VDW.get(p[\"elem\"],1.7) for p in prot])[None,:]) - d, 0, None)\n        bad = (ov > 0.4).any(axis=1)\n        ax.scatter(Y[~bad, 0], Y[~bad, 1], s=90, color=SECOND, label=\"your pose\")\n        if bad.any():\n            ax.scatter(Y[bad, 0], Y[bad, 1], s=130, color=\"#DC2626\", marker=\"X\",\n                       label=\"clashing atom\")\n        ax.set_aspect(\"equal\"); ax.legend(fontsize=8, loc=\"best\")\n        ax.set_xlabel(\"x (A)\"); ax.set_ylabel(\"y (A)\")\n        ax.set_title(f\"score {sc['score']:+.1f}    RMSD to crystal {r:.2f} A\")\n        plt.tight_layout(); plt.show()\n    print(f\"score            {sc['score']:+8.2f}   (more negative is better)\")\n    print(f\"  contact        {sc['contact']:8.2f}\")\n    print(f\"  hydrogen bonds {sc['hbond']:8.2f}\")\n    print(f\"  clash penalty  {sc['clash']:8.2f}   from {sc['clashing_atoms']} atoms\")\n    print(f\"RMSD to crystal  {r:8.2f} A\")\n    return dict(**sc, rmsd=round(r, 2))\n\n\ndef score_vs_truth(n=220, spread=4.0, seed=7, resn=\"THA\", show=True):\n    \"\"\"Generate real perturbed poses; plot score against RMSD to the crystal.\"\"\"\n    rng = np.random.default_rng(seed)\n    atoms, X = ligand(resn)\n    rows = []\n    for _ in range(n):\n        t = rng.normal(0, spread / 2, 3)\n        a = rng.uniform(-180, 180)\n        ax_ = \"xyz\"[rng.integers(0, 3)]\n        Y = move_ligand(*t, rot_deg=a, axis=ax_, resn=resn)\n        sc = score_pose(Y, resn)\n        rows.append((rmsd(X, Y), sc[\"score\"], sc[\"clashing_atoms\"]))\n    R = np.array([r[0] for r in rows]); S = np.array([r[1] for r in rows])\n    good_rmsd = R < 2.0\n    best = int(np.argmin(S))\n    if show:\n        f, ax = _fig(9.0, 5.0)\n        ax.scatter(R[~good_rmsd], S[~good_rmsd], s=26, color=MUTED,\n                   label=\"wrong pose (RMSD >= 2 A)\", alpha=.75)\n        ax.scatter(R[good_rmsd], S[good_rmsd], s=40, color=PRIMARY,\n                   label=\"correct pose (RMSD < 2 A)\")\n        ax.scatter([R[best]], [S[best]], s=150, marker=\"*\", color=SECOND,\n                   zorder=5, label=\"best-scoring pose\")\n        lo = float(S.min()); hi = float(np.percentile(S, 55))\n        pad = max(10.0, 0.10 * (hi - lo)); off = int((S > hi + pad).sum())\n        ax.set_ylim(lo - pad, hi + pad)\n        if off:\n            ax.text(0.99, 0.03, f\"{off} heavily clashing poses lie above this axis\",\n                    transform=ax.transAxes, ha=\"right\", va=\"bottom\",\n                    fontsize=8, color=MUTED)\n        ax.axvline(2.0, color=\"#B45309\", ls=\":\", lw=1.4)\n        ax.set_xlabel(\"RMSD to the crystal pose (A)\")\n        ax.set_ylabel(\"score (more negative = better)\")\n        ax.legend(fontsize=8)\n        ax.set_title(f\"{n} real poses of tacrine in the real gorge\")\n        plt.tight_layout(); plt.show()\n    picked_right = bool(R[best] < 2.0)\n    print(f\"{n} poses generated from the real crystal coordinates\")\n    print(f\"  poses within 2 A of the crystal : {int(good_rmsd.sum())}\")\n    print(f\"  best-scoring pose has RMSD      : {R[best]:.2f} A\")\n    print(f\"  did the score pick a correct pose? {'yes' if picked_right else 'NO'}\")\n    return dict(n=n, correct_available=int(good_rmsd.sum()),\n                best_rmsd=round(float(R[best]), 2),\n                best_score=round(float(S[best]), 2),\n                score_picked_correct=picked_right)\n\n\n# ---------------------------------------------------------------- L27\ndef empty_space(probe=1.6, grid=1.0, around=9.0, resn=\"THA\", show=True):\n    \"\"\"Find unfilled volume in the real gorge next to the bound ligand.\n\n    Only gorge atoms are considered and the grid is processed in chunks, so a\n    fine grid stays within memory.\n    \"\"\"\n    if not isinstance(resn, str):\n        raise TypeError(\"resn must be a residue name such as 'THA'; \"\n                        \"did you mean empty_space(probe=%r)?\" % resn)\n    if grid < 0.3:\n        raise ValueError(\"grid finer than 0.3 A is not useful here and is slow\")\n    atoms, X = ligand(resn)\n    prot, Pall = protein()\n    c = X.mean(axis=0)\n    # prune to atoms that could possibly matter\n    keep_p = np.linalg.norm(Pall - c, axis=1) <= (around + probe + 2.5)\n    P = Pall[keep_p]\n    rp = np.array([VDW.get(r[\"elem\"], 1.7) for r in np.array(prot)[keep_p]])[None, :]\n    rl = np.array([VDW.get(a[\"elem\"], 1.7) for a in atoms])[None, :]\n    rng_ = np.arange(-around, around + 1e-9, grid)\n    G = np.array([[x, y, z] for x in rng_ for y in rng_ for z in rng_]) + c\n    G = G[np.linalg.norm(G - c, axis=1) <= around]\n    free = np.empty(len(G), bool)\n    dlmin = np.empty(len(G))\n    notlig = np.empty(len(G), bool)\n    step = max(1, int(4e6 // max(len(P), 1)))\n    for i in range(0, len(G), step):\n        g = G[i:i + step]\n        dp = np.linalg.norm(g[:, None, :] - P[None, :, :], axis=2)\n        free[i:i + step] = (dp > (rp + probe)).all(axis=1)\n        dl = np.linalg.norm(g[:, None, :] - X[None, :, :], axis=2)\n        notlig[i:i + step] = (dl > (rl + probe)).all(axis=1)\n        dlmin[i:i + step] = dl.min(axis=1)\n    pocket = free & (dlmin < 8.0)\n    unfilled = pocket & notlig\n    if show:\n        zc = float(X[:, 2].mean()); slab = 2.0\n        sl = np.abs(G[:, 2] - zc) <= slab\n        lig = np.abs(X[:, 2] - zc) <= slab\n        f, ax = _fig(7.0, 6.0)\n        ax.scatter(G[pocket & ~notlig & sl, 0], G[pocket & ~notlig & sl, 1],\n                   s=9, color=\"#CBD5E1\", label=\"taken up by the ligand\")\n        ax.scatter(G[unfilled & sl, 0], G[unfilled & sl, 1], s=12, color=OK,\n                   label=f\"free, probe {probe} A\")\n        ax.scatter(X[lig, 0], X[lig, 1], s=80, color=SECOND, label=\"tacrine\")\n        ax.set_aspect(\"equal\"); ax.legend(fontsize=8, loc=\"upper right\")\n        ax.set_xlabel(\"x (A)\"); ax.set_ylabel(\"y (A)\")\n        ax.set_title(f\"a {2*slab:.0f} A slice through the gorge, not the whole \"\n                     f\"volume\\n{int((unfilled & sl).sum())} of the \"\n                     f\"{int(unfilled.sum())} free points lie in this slice\",\n                     fontsize=10)\n        plt.tight_layout(); plt.show()\n    vol = float(unfilled.sum()) * grid ** 3\n    print(f\"probe radius             {probe} A\")\n    print(f\"grid points sampled      {len(G):,} at {grid} A spacing\")\n    print(f\"free and near the ligand {unfilled.sum():>8,}  -> about {vol:,.0f} A^3\")\n    print(\"\\n  That volume is what a de novo program would try to fill, and why\")\n    print(\"  a first design should NOT fill all of it.\")\n    return dict(probe=probe, grid=grid, grid_points=len(G),\n                unfilled=int(unfilled.sum()), volume_A3=round(vol, 1))\n\n\ndef score_ranking(n=220, spread=2.0, seed=7, top=10, resn=\"THA\"):\n    \"\"\"Rank real poses by score and show the RMSD of each. The question is not\n    only whether the top pose is right but how well the score SEPARATES.\"\"\"\n    rng = np.random.default_rng(seed)\n    atoms, X = ligand(resn)\n    rows = []\n    for _ in range(n):\n        t = rng.normal(0, spread / 2, 3)\n        Y = move_ligand(*t, rot_deg=rng.uniform(-180, 180),\n                        axis=\"xyz\"[rng.integers(0, 3)], resn=resn)\n        rows.append((rmsd(X, Y), score_pose(Y, resn)[\"score\"]))\n    rows.sort(key=lambda r: r[1])\n    print(f\"top {top} of {n} poses, ranked by score\")\n    print(f\"  {'rank':>4}  {'score':>9}  {'RMSD':>6}   verdict\")\n    for i, (r, sc) in enumerate(rows[:top], 1):\n        print(f\"  {i:>4}  {sc:>9.2f}  {r:>5.2f} A   \"\n              f\"{'correct' if r < 2 else 'WRONG POSE'}\")\n    wrong = sum(1 for r, _ in rows[:top] if r >= 2)\n    print(f\"\\n  {wrong} of the top {top} scoring poses are the wrong pose.\")\n    return dict(top=top, wrong_in_top=wrong,\n                rows=[(round(r, 2), round(s, 2)) for r, s in rows[:top]])\n\n\n_RING = {\"TRP\": {\"CD2\", \"CE2\", \"CE3\", \"CZ2\", \"CZ3\", \"CH2\"},\n         \"PHE\": {\"CG\", \"CD1\", \"CD2\", \"CE1\", \"CE2\", \"CZ\"},\n         \"TYR\": {\"CG\", \"CD1\", \"CD2\", \"CE1\", \"CE2\", \"CZ\"},\n         \"HIS\": {\"CG\", \"ND1\", \"CD2\", \"CE1\", \"NE2\"}}\n\n\ndef stacking(near=6.0, resn=\"THA\", show=True):\n    \"\"\"Is the ligand sandwiched? Measure each nearby aromatic ring's position\n    relative to the plane of the ligand itself.\"\"\"\n    atoms, X = ligand(resn)\n    prot, P = protein()\n    cen = X.mean(axis=0)\n    nrm = np.linalg.svd(X - cen)[2][2]\n    seen, rows = set(), []\n    for j, r in enumerate(prot):\n        kk = (r[\"resn\"], r[\"resi\"])\n        if r[\"resn\"] not in _RING or kk in seen:\n            continue\n        idx = [k for k, q in enumerate(prot)\n               if q[\"resn\"] == r[\"resn\"] and q[\"resi\"] == r[\"resi\"]\n               and q[\"name\"] in _RING[r[\"resn\"]]]\n        if len(idx) < 5:\n            continue\n        Q = P[idx]\n        qc = Q.mean(axis=0)\n        if np.linalg.norm(qc - cen) > near + 3.0:\n            continue\n        seen.add(kk)\n        qn = np.linalg.svd(Q - qc)[2][2]\n        signed = float(np.dot(qc - cen, nrm))\n        ang = float(np.degrees(np.arccos(min(1.0, abs(float(np.dot(nrm, qn)))))))\n        rows.append((f\"{r['resn']}{r['resi']}\", signed,\n                     float(np.linalg.norm(qc - cen)), ang))\n    rows.sort(key=lambda t: t[2])\n    rows = [t for t in rows if t[2] <= near + 1.0]\n    if show:\n        f, ax = _fig(7.6, 4.4)\n        for nm, sg, dd, an in rows:\n            st = an < 30\n            ax.barh(nm, sg, height=.6, color=(PRIMARY if sg > 0 else SECOND),\n                    alpha=(1.0 if st else 0.35),\n                    hatch=(None if st else \"///\"), edgecolor=\"white\")\n            ax.text(sg + (0.2 if sg > 0 else -0.2), nm,\n                    f\"{an:.0f} deg\" + (\"\" if st else \"  edge-on\"),\n                    va=\"center\", ha=(\"left\" if sg > 0 else \"right\"),\n                    fontsize=8, color=(\"#1F2937\" if st else MUTED))\n        ax.set_xlim(min(r[1] for r in rows) - 3.0, max(r[1] for r in rows) + 3.0)\n        ax.axvline(0, color=\"#1F2937\", lw=1.4)\n        ax.set_xlabel(\"signed distance from the ligand's plane (A)\")\n        ax.set_title(\"opposite sides of zero = sandwiched;\\nsolid bar = near-parallel = genuinely stacked\", fontsize=10)\n        plt.tight_layout(); plt.show()\n    print(\"aromatic rings near the ligand, measured against the ligand's own plane\")\n    print(f\"  {'residue':<9}{'side':>6}{'perp (A)':>10}{'centroid (A)':>14}\"\n          f\"{'planes (deg)':>14}\")\n    for nm, sg, dd, an in rows:\n        print(f\"  {nm:<9}{('above' if sg > 0 else 'below'):>6}{sg:>10.2f}\"\n              f\"{dd:>14.2f}{an:>14.1f}\")\n    above = [r for r in rows if r[1] > 0]\n    below = [r for r in rows if r[1] < 0]\n    print(f\"\\n  {len(above)} above the plane, {len(below)} below.\")\n    if above and below:\n        print(\"  Rings on both sides: the ligand is sandwiched.\")\n    print(\"  A plane angle under about 30 degrees means the rings are \"\n          \"roughly parallel,\\n  which is face-to-face stacking rather than \"\n          \"edge-on contact.\")\n    return dict(rows=[(n, round(s, 2), round(d, 2), round(a, 1))\n                      for n, s, d, a in rows],\n                above=len(above), below=len(below),\n                sandwiched=bool(above and below))\n", "crystal.py", "exec"), _cr.__dict__)
sys.modules["crystal"] = _cr
crystal = _cr

_sm = types.ModuleType("sims"); _sm.__dict__["__name__"] = "sims"
exec(compile("\"\"\"PHS 312 simulations. NumPy + matplotlib only - no install, no network.\n\nEvery function returns the numbers it plotted so the notebook can quiz on them.\n\"\"\"\nimport numpy as np\nimport matplotlib\nimport matplotlib.pyplot as plt\n\nPRIMARY, SECOND, MUTED = \"#0B6E8F\", \"#C2410C\", \"#6B7280\"\n\n\ndef _tw(text, width=76, indent=\"\"):\n    import textwrap\n    return \"\\n\".join(textwrap.fill(p, width, initial_indent=indent,\n                                   subsequent_indent=indent)\n                     for p in text.split(\"\\n\"))\n\n\ndef _fig(w=9.5, h=4.4):\n    f, ax = plt.subplots(figsize=(w, h))\n    ax.grid(alpha=.25); ax.set_axisbelow(True)\n    for s in (\"top\", \"right\"): ax.spines[s].set_visible(False)\n    return f, ax\n\n\n# ---------------------------------------------------------------- L22\ndef force_field_terms(bond_k=300., bond_r0=1.52, ang_k=60., ang_t0=109.5,\n                      tor_V=14., vdw_eps=0.3, vdw_sigma=3.6, q1q2=-0.3):\n    \"\"\"Each force-field term plotted over its own coordinate.\"\"\"\n    f, axes = plt.subplots(1, 4, figsize=(15, 3.4))\n    r = np.linspace(1.1, 2.1, 200); Es = bond_k * (r - bond_r0) ** 2\n    t = np.linspace(80, 140, 200);  Eb = ang_k * np.radians(t - ang_t0) ** 2\n    d = np.linspace(-180, 180, 361); Et = tor_V / 2 * (1 + np.cos(np.radians(3 * d)))\n    x = np.linspace(3.0, 8.0, 300)\n    Ev = 4 * vdw_eps * ((vdw_sigma / x) ** 12 - (vdw_sigma / x) ** 6)\n    Ec = 332. * q1q2 / x\n    for ax, (xx, yy, xl, ti) in zip(axes, [\n            (r, Es, \"bond length (A)\", \"stretch\"),\n            (t, Eb, \"bond angle (deg)\", \"bend\"),\n            (d, Et, \"torsion (deg)\", \"torsion\"),\n            (x, Ev + Ec, \"separation (A)\", \"vdW + electrostatic\")]):\n        ax.plot(xx, yy, color=PRIMARY, lw=2)\n        ax.set_title(ti, fontsize=10); ax.set_xlabel(xl, fontsize=8)\n        ax.grid(alpha=.25); ax.axhline(0, color=MUTED, lw=.8)\n        for s in (\"top\", \"right\"): ax.spines[s].set_visible(False)\n    axes[0].set_ylabel(\"energy (kcal/mol)\", fontsize=9)\n    plt.tight_layout(); plt.show()\n    return dict(stretch_min=float(r[Es.argmin()]), torsion_barrier=float(Et.max()),\n                vdw_min_sep=float(x[(Ev + Ec).argmin()]),\n                vdw_min_E=float((Ev + Ec).min()))\n\n\n# ---------------------------------------------------------------- L23\ndef torsion_profile(V3=14.0, gauche_strain=3.8, step=10, start_angle=100.):\n    \"\"\"Butane-like relaxed torsion profile, plus where minimisation stops.\"\"\"\n    d = np.arange(-180, 181, step)\n    E = V3 / 2 * (1 + np.cos(np.radians(3 * d)))\n    E += gauche_strain * np.exp(-((d - 60) / 32.) ** 2)\n    E += gauche_strain * np.exp(-((d + 60) / 32.) ** 2)\n    E += 2.2 * gauche_strain * np.exp(-(d / 30.) ** 2)\n    E = E - E.min()\n    # steepest descent from start_angle on this surface\n    fine = np.arange(-180, 180.01, 0.5)\n    Ef = np.interp(fine, d, E)\n    i = int(np.argmin(np.abs(fine - start_angle)))\n    while True:\n        lo, hi = max(i - 1, 0), min(i + 1, len(fine) - 1)\n        j = lo if Ef[lo] < Ef[i] else (hi if Ef[hi] < Ef[i] else i)\n        if j == i: break\n        i = j\n    f, ax = _fig()\n    ax.plot(d, E, \"-o\", ms=3.5, color=PRIMARY, label=\"relaxed scan\")\n    ax.axvline(start_angle, color=MUTED, ls=\":\", label=f\"start {start_angle:.0f}\")\n    ax.plot([fine[i]], [Ef[i]], \"v\", ms=12, color=SECOND,\n            label=f\"minimisation stops at {fine[i]:.0f} deg\")\n    ax.set_xlabel(\"C1-C2-C3-C4 dihedral (degrees)\")\n    ax.set_ylabel(\"relative steric energy (kJ/mol)\")\n    ax.set_xticks(range(-180, 181, 60)); ax.legend(fontsize=8)\n    plt.tight_layout(); plt.show()\n    mins = [float(d[k]) for k in range(1, len(d) - 1)\n            if E[k] < E[k - 1] and E[k] < E[k + 1]]\n    return dict(global_min=float(d[E.argmin()]), local_minima=mins,\n                barrier_at_0=float(E[d == 0][0]), stops_at=float(fine[i]))\n\n\ndef search_cost(n_rot=7, step=30):\n    n = int(round(360 / step)) ** n_rot\n    secs = n * 0.05\n    return dict(conformations=n, hours=secs / 3600,\n                years=secs / 3600 / 24 / 365)\n\n\n# ---------------------------------------------------------------- L24\ndef anneal(T=300., n_steps=4000, seed=0):\n    \"\"\"Metropolis walk on a double-well surface: does it escape the local min?\n\n    Global minima sit at +/-180 deg (0 kJ/mol); a local minimum sits at\n    +60 deg (4 kJ/mol) behind a ~26 kJ/mol barrier. At low T the walker is\n    trapped; raising T lets it cross.\n    \"\"\"\n    rng = np.random.default_rng(seed)\n    PLATEAU, DEEP, SHALLOW, W = 30.0, 30.0, 26.0, 26.0\n\n    def E(a):\n        a = np.asarray(a, float)\n        return (PLATEAU\n                - DEEP * np.exp(-((a - 180) / W) ** 2)\n                - DEEP * np.exp(-((a + 180) / W) ** 2)\n                - SHALLOW * np.exp(-((a - 60) / W) ** 2))\n\n    pos = 60.0\n    visited = [pos]\n    kT = max(T, 1.) * 0.0083145                      # kJ/mol per K\n    for _ in range(n_steps):\n        trial = ((pos + rng.normal(0, 20) + 180) % 360) - 180\n        dE = float(E(trial) - E(pos))\n        if dE < 0 or rng.random() < np.exp(-min(dE / kT, 700)):\n            pos = trial\n        visited.append(pos)\n    visited = np.array(visited)\n\n    second_half = visited[len(visited) // 2:]\n    near_global = float(np.mean(np.abs(np.abs(second_half) - 180) < 50) * 100)\n    ever = bool(np.any(np.abs(np.abs(visited) - 180) < 50))\n\n    x = np.linspace(-180, 180, 721)\n    f, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))\n    a1.plot(x, E(x), color=PRIMARY, lw=2)\n    a1.set_xlabel(\"dihedral (deg)\"); a1.set_ylabel(\"energy (kJ/mol)\")\n    a1.set_title(\"energy surface\")\n    a1.plot([60], [E(60)], \"o\", color=SECOND, ms=10)\n    a1.annotate(\"start (local, 4 kJ/mol)\", (60, E(60) + 1.5), fontsize=8, ha=\"center\")\n    a1.annotate(\"global (0 kJ/mol)\", (180, E(180) + 1.5), fontsize=8, ha=\"right\")\n    a2.hist(visited, bins=72, color=PRIMARY, alpha=.85)\n    a2.set_xlabel(\"dihedral visited (deg)\"); a2.set_ylabel(\"count\")\n    a2.set_title(f\"T = {T:.0f} K   -   \"\n                 + (\"escaped the local minimum\" if near_global > 15\n                    else \"stayed trapped\"))\n    for ax in (a1, a2):\n        ax.grid(alpha=.25)\n        for sp in (\"top\", \"right\"): ax.spines[sp].set_visible(False)\n    plt.tight_layout(); plt.show()\n    return dict(escaped=near_global > 15, pct_near_global=round(near_global, 1),\n                ever_reached_global=ever, final=round(float(visited[-1]), 1))\n\n\ndef rmsd_overlay(dx=0.0, dy=0.0, rot_deg=0.0):\n    \"\"\"Rigid-body move of a probe onto a template; reports RMSD.\"\"\"\n    tmpl = np.array([[0., 0.], [1.4, 0.3], [2.6, -0.4], [3.9, 0.2], [1.2, 1.7]])\n    th = np.radians(rot_deg)\n    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])\n    probe = (tmpl - tmpl.mean(0)) @ R.T + tmpl.mean(0) + np.array([dx, dy])\n    rmsd = float(np.sqrt(((probe - tmpl) ** 2).sum(1).mean()))\n    f, ax = _fig(6.4, 4.6)\n    ax.plot(*tmpl.T, \"o-\", color=PRIMARY, ms=11, lw=2, label=\"template (rigid active)\")\n    ax.plot(*probe.T, \"s--\", color=SECOND, ms=10, lw=2, label=\"your analogue\")\n    for p, q in zip(tmpl, probe):\n        ax.plot([p[0], q[0]], [p[1], q[1]], color=MUTED, lw=.9, ls=\":\")\n    ax.set_title(f\"RMSD = {rmsd:.2f} A\"); ax.legend(fontsize=8); ax.set_aspect(\"equal\")\n    plt.tight_layout(); plt.show()\n    return dict(rmsd=rmsd, good=rmsd < 0.5)\n\n\n# ---------------------------------------------------------------- L25\n# A snug pocket, 12.0 x 8.0 A, holding a ligand that spans 5.0 A, with a\n# gap in the wall where a polar residue sits. Orientation matters: turned\n# across the pocket the ligand cannot fit without clashing.\n_A, _B = 6.0, 4.1\n_TH = np.linspace(0, 2 * np.pi, 25)[:-1]\n_TH = np.array([t for t in _TH\n                if abs(np.degrees(t) - 90) > 16])          # gap for the residue\nSITE = np.c_[_A * np.cos(_TH), _B * np.sin(_TH)]           # cavity wall atoms\nHBOND_POINT = np.array([0.0, 3.7])                         # polar partner\nLIGAND = np.array([[-2.5, -0.3], [-0.8, 0.3], [0.9, 0.2],\n                   [2.5, -0.4], [0.0, 1.3]])               # last atom is polar\nCLASH_AT, CONTACT_AT, HBOND_AT = 3.0, 3.9, 2.7\n\n\ndef dock_pose(x=0.0, y=0.0, rot_deg=0.0, show=True):\n    \"\"\"Place a rigid 5-atom ligand in a 2-D pocket it very nearly fills.\n\n    Score = -(shape contacts + H-bond) + steric clash penalty, so a MORE\n    NEGATIVE score is a better pose. A pose outside the pocket earns nothing\n    and tends to zero; a pose driven into the wall goes sharply positive.\n    \"\"\"\n    th = np.radians(rot_deg)\n    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])\n    pose = LIGAND @ R.T + np.array([x, y])\n\n    d = np.linalg.norm(pose[:, None, :] - SITE[None, :, :], axis=2)\n    nearest = d.min(1)\n    bad = nearest < CLASH_AT\n    clash = float(np.sum(np.clip(CLASH_AT - nearest, 0, None) ** 2) * 12)\n    contact = float(np.sum(np.exp(-((nearest - CONTACT_AT) / 1.25) ** 2)) * 2.6)\n    hb_dist = float(np.linalg.norm(pose[4] - HBOND_POINT))\n    hbond = float(6.0 * np.exp(-((hb_dist - HBOND_AT) / 0.95) ** 2))\n    score = -(contact + hbond) + clash\n\n    if show:\n        f, ax = _fig(7.0, 5.4)\n        _full = np.linspace(0, 2 * np.pi, 160)\n        ax.fill(_A * np.cos(_full), _B * np.sin(_full), color=\"#DCE9EF\",\n                zorder=0)\n        ax.plot(np.append(SITE[:, 0], SITE[0, 0]),\n                np.append(SITE[:, 1], SITE[0, 1]), \"o\", color=PRIMARY,\n                ms=15, label=\"binding-site wall\", zorder=1)\n        ax.plot(*HBOND_POINT, \"*\", ms=22, color=\"#B45309\",\n                label=\"polar residue\", zorder=3)\n        chain = pose[:4]\n        ax.plot(chain[:, 0], chain[:, 1], \"-\", color=SECOND, lw=2.6,\n                alpha=.8, zorder=4)\n        ax.plot([chain[1, 0], pose[4, 0]], [chain[1, 1], pose[4, 1]], \"-\",\n                color=SECOND, lw=2.6, alpha=.8, zorder=4)\n        ok = ~bad\n        if ok.any():\n            ax.plot(pose[ok][:, 0], pose[ok][:, 1], \"o\", ms=14,\n                    color=SECOND, label=\"ligand atom\", zorder=5)\n        if bad.any():\n            ax.plot(pose[bad][:, 0], pose[bad][:, 1], \"X\", ms=19,\n                    color=\"#DC2626\", label=\"steric clash\", zorder=6)\n        ax.set_aspect(\"equal\")\n        ax.set_xlim(-9.5, 9.5); ax.set_ylim(-7, 7)\n        ax.legend(fontsize=8, loc=\"lower left\")\n        ax.set_title(f\"score {score:+.1f}    \"\n                     f\"contacts {contact:.1f}   H-bond {hbond:.1f}   \"\n                     f\"clash {clash:.1f}\")\n        plt.tight_layout(); plt.show()\n    return dict(score=round(score, 2), contact=round(contact, 2),\n                hbond=round(hbond, 2), clash=round(clash, 2),\n                clashing_atoms=int(bad.sum()))\n\n\n# ---------------------------------------------------------------- L26\ndef score_vs_truth(cutoff=-7.0, seed=3):\n    \"\"\"Docking scores against crystallographic truth: the ranking problem.\"\"\"\n    rng = np.random.default_rng(seed)\n    n = 60\n    active = rng.random(n) < 0.25\n    rmsd = np.where(active, rng.gamma(2, .9, n), rng.gamma(5, 1.1, n))\n    score = -4.5 - 3.2 * active - rng.normal(0, 1.5, n)\n    picked = score <= cutoff\n    tp = int(np.sum(picked & active)); fp = int(np.sum(picked & ~active))\n    tot_act = int(active.sum())\n    hit = 100 * tp / max(picked.sum(), 1)\n    base = 100 * tot_act / n\n    f, ax = _fig(8.6, 4.6)\n    ax.scatter(score[~active], rmsd[~active], s=46, color=MUTED,\n               label=\"decoy\", alpha=.75)\n    ax.scatter(score[active], rmsd[active], s=56, color=PRIMARY,\n               label=\"true active\", alpha=.9)\n    ax.axvline(cutoff, color=SECOND, lw=2, ls=\"--\",\n               label=f\"your cutoff {cutoff:+.1f}\")\n    ax.axhline(2.0, color=\"#B45309\", lw=1.2, ls=\":\",\n               label=\"2 A - pose counts as correct\")\n    ax.set_xlabel(\"docking score (more negative = better)\")\n    ax.set_ylabel(\"RMSD to crystal pose (A)\")\n    ax.legend(fontsize=8)\n    ax.set_title(f\"picked {int(picked.sum())} compounds: {tp} actives, {fp} decoys\"\n                 f\"   |   hit rate {hit:.0f}% vs {base:.0f}% at random\")\n    plt.tight_layout(); plt.show()\n    return dict(picked=int(picked.sum()), true_actives_found=tp, decoys=fp,\n                total_actives=tot_act, hit_rate=round(hit, 1),\n                random_rate=round(base, 1),\n                enrichment=round(hit / base, 2) if base else 0.0)\n\n\n# ---------------------------------------------------------------- L27\ndef de_novo(h_bond_group=\"amidine\", near_group=\"phenyl\", far_group=\"isopropyl\",\n            linker_bonds=5, fill_site=False):\n    \"\"\"Grow and link fragments into a site with one polar and two greasy pockets.\"\"\"\n    HB = {\"amidine\": 5.0, \"tetrazole\": 4.2, \"phenol\": 3.0, \"methyl\": 0.2,\n          \"isopropyl\": 0.0}\n    GREASE = {\"phenyl\": 3.6, \"cyclohexyl\": 3.9, \"isopropyl\": 2.6,\n              \"methyl\": 1.2, \"amidine\": -1.5, \"tetrazole\": -1.8, \"phenol\": 0.6}\n    SYNTH = {\"amidine\": 1.0, \"tetrazole\": 1.6, \"phenol\": 0.3, \"phenyl\": 0.2,\n             \"cyclohexyl\": 0.5, \"isopropyl\": 0.1, \"methyl\": 0.0}\n    inter = HB.get(h_bond_group, 0) + GREASE.get(near_group, 0) + GREASE.get(far_group, 0)\n    ideal = 6\n    strain = 0.9 * abs(linker_bonds - ideal)\n    synth = SYNTH.get(h_bond_group, 0) + SYNTH.get(near_group, 0) + SYNTH.get(far_group, 0) \\\n        + 0.25 * max(linker_bonds - 8, 0)\n    overfill = 3.0 if fill_site else 0.0\n    total = inter - strain - synth - overfill\n    labels = [\"H-bond region\", \"near pocket\", \"far pocket\",\n              \"linker strain\", \"synthetic cost\", \"over-fill\"]\n    vals = [HB.get(h_bond_group, 0), GREASE.get(near_group, 0),\n            GREASE.get(far_group, 0), -strain, -synth, -overfill]\n    f, ax = _fig(9.2, 3.8)\n    cols = [PRIMARY if v >= 0 else SECOND for v in vals]\n    ax.barh(labels, vals, color=cols)\n    ax.axvline(0, color=\"k\", lw=.9)\n    ax.set_xlabel(\"contribution to design score\")\n    ax.set_title(f\"design score {total:+.1f}\"\n                 + (\"   (over-filled: no room left to optimise)\" if fill_site else \"\"))\n    plt.tight_layout(); plt.show()\n    return dict(score=round(total, 2), interactions=round(inter, 2),\n                strain=round(strain, 2), synthetic_cost=round(synth, 2),\n                overfill_penalty=overfill)\n\n\n# ---------------------------------------------------------------- L29 / L30\nCOUMARIN = dict(\n    X=[\"H\", \"Me\", \"OH\", \"OMe\", \"OCF3\", \"NH2\", \"F\", \"Cl\", \"CF3\", \"CN\", \"NO2\"],\n    pIC50=np.array([8.36, 8.38, 8.01, 8.44, 7.94, 7.46, 8.55, 8.48, 8.24, 7.97, 8.59]),\n    pi=np.array([0., .49, -.67, -.09, 1.02, -1.23, .14, .71, .88, -.57, -.26]),\n    Vdw=np.array([.08, 1.01, .49, 1.49, 1.60, .67, .36, 1.07, 1.11, 1.14, 1.20]),\n    F=np.array([0., -.04, .28, .25, .37, .02, .42, .40, .37, .50, .66]))\n\n_PI = {\"H\": 0., \"Me\": .56, \"Cl\": .71, \"NO2\": -.28, \"OMe\": -.02, \"NH2\": -1.23, \"F\": .14}\n_MR = {\"H\": .103, \"Me\": .565, \"Cl\": .603, \"NO2\": .736, \"OMe\": .787, \"NH2\": .542, \"F\": .092}\n_F = {\"H\": 0., \"Me\": .01, \"Cl\": .42, \"NO2\": .65, \"OMe\": .29, \"NH2\": .08, \"F\": .45}\n_R = {\"H\": 0., \"Me\": -.18, \"Cl\": -.19, \"NO2\": .13, \"OMe\": -.56, \"NH2\": -.74, \"F\": -.39}\n_TAC = [(\"tacrine\", \"H\", \"H\", 6.59), (\"I\", \"Me\", \"H\", 5.09), (\"II\", \"H\", \"Me\", 7.00),\n        (\"III\", \"Cl\", \"H\", 6.26), (\"IV\", \"H\", \"Cl\", 8.00), (\"V\", \"NO2\", \"H\", 5.52),\n        (\"VI\", \"H\", \"NO2\", 7.55), (\"VII\", \"H\", \"OMe\", 6.45), (\"VIII\", \"NH2\", \"H\", 5.42),\n        (\"IX\", \"H\", \"F\", 7.06), (\"X\", \"Cl\", \"Cl\", 6.32), (\"XI\", \"OMe\", \"OMe\", 5.29)]\n\ndef tacrine_table():\n    rows = []\n    for cid, X, Y, p in _TAC:\n        rows.append(dict(cpd=cid, X=X, Y=Y, pIC50=p, piX=_PI[X], piY=_PI[Y],\n                         MRX=_MR[X], MRY=_MR[Y], F_XY=_F[X] + _F[Y], R_XY=_R[X] + _R[Y]))\n    return rows\n\n\ndef _lstsq(X, y):\n    X = np.column_stack([np.asarray(X, float).reshape(len(y), -1), np.ones(len(y))])\n    beta, *_ = np.linalg.lstsq(X, y, rcond=None)\n    pred = X @ beta\n    ss_res = float(((y - pred) ** 2).sum()); ss_tot = float(((y - y.mean()) ** 2).sum())\n    r2 = 1 - ss_res / ss_tot\n    n, p = len(y), X.shape[1]\n    s = float(np.sqrt(ss_res / (n - p))) if n > p else float(\"nan\")\n    return beta, pred, r2, s\n\n\ndef qsar_single(param=\"pi\", quadratic=False):\n    \"\"\"One descriptor at a time on the coumarin MAO-B series.\"\"\"\n    y = COUMARIN[\"pIC50\"]; x = COUMARIN[param]\n    cols = np.column_stack([x, x ** 2]) if quadratic else x\n    beta, pred, r2, s = _lstsq(cols, y)\n    f, ax = _fig(7.6, 4.4)\n    ax.scatter(x, y, s=70, color=PRIMARY, zorder=3)\n    for xi, yi, t in zip(x, y, COUMARIN[\"X\"]):\n        ax.annotate(t, (xi, yi), fontsize=8, xytext=(4, 4), textcoords=\"offset points\")\n    xs = np.linspace(x.min(), x.max(), 200)\n    ys = beta[0] * xs + beta[1] * xs ** 2 + beta[2] if quadratic else beta[0] * xs + beta[1]\n    ax.plot(xs, ys, color=SECOND, lw=2,\n            label=(\"quadratic\" if quadratic else \"linear\") + f\"   r = {np.sqrt(max(r2,0)):.3f}   s = {s:.3f}\")\n    ax.set_xlabel({\"pi\": \"substituent hydrophobicity  pi\",\n                   \"Vdw\": \"van der Waals volume\", \"F\": \"inductive factor  F\"}[param])\n    ax.set_ylabel(\"pIC50 (MAO-B)\"); ax.legend(fontsize=9)\n    plt.tight_layout(); plt.show()\n    return dict(param=param, quadratic=quadratic,\n                r=round(float(np.sqrt(max(r2, 0))), 3), s=round(s, 3))\n\n\ndef qsar_multi(use=(\"MRX\",), show=True):\n    \"\"\"Add descriptors and watch r rise while leave-one-out gets worse.\"\"\"\n    rows = tacrine_table()\n    y = np.array([r[\"pIC50\"] for r in rows])\n    X = np.column_stack([[r[c] for r in rows] for c in use])\n    beta, pred, r2, s = _lstsq(X, y)\n    loo = []\n    for i in range(len(y)):\n        m = np.ones(len(y), bool); m[i] = False\n        b, _, _, _ = _lstsq(X[m], y[m])\n        loo.append(float(np.dot(np.append(X[i], 1.0), b)))\n    loo = np.array(loo)\n    q2 = 1 - float(((y - loo) ** 2).sum()) / float(((y - y.mean()) ** 2).sum())\n    if show:\n        f, ax = _fig(5.6, 5.2)\n        ax.scatter(pred, y, s=72, color=PRIMARY, label=\"fitted\", zorder=3)\n        ax.scatter(loo, y, s=52, color=SECOND, marker=\"s\",\n                   label=\"left-out prediction\", alpha=.85, zorder=3)\n        lim = [min(y.min(), pred.min(), loo.min()) - .4,\n               max(y.max(), pred.max(), loo.max()) + .4]\n        ax.plot(lim, lim, \"k--\", lw=1); ax.set_xlim(lim); ax.set_ylim(lim)\n        ax.set_xlabel(\"predicted pIC50\"); ax.set_ylabel(\"observed pIC50\")\n        ax.set_title(\" + \".join(use) + f\"\\nr = {np.sqrt(max(r2,0)):.3f}    \"\n                     f\"s = {s:.3f}    q2 = {q2:.3f}\")\n        ax.legend(fontsize=8); plt.tight_layout(); plt.show()\n    return dict(descriptors=list(use), n=len(y), k=len(use),\n                r=round(float(np.sqrt(max(r2, 0))), 3), s=round(s, 3),\n                q2=round(q2, 3), compounds_per_parameter=round(len(y) / len(use), 1))\n\n\n# ---------------------------------------------------------------- L31\nBIOISOSTERE = {\n    \"carboxylic acid (COOH)\": dict(pKa=4.2, logP=0.0, MW=45, HBD=1, HBA=2, note=\"parent\"),\n    \"tetrazole\":              dict(pKa=4.9, logP=0.5, MW=69, HBD=1, HBA=4, note=\"classic acid mimic\"),\n    \"acylsulfonamide\":        dict(pKa=4.5, logP=0.6, MW=121, HBD=1, HBA=4, note=\"acid mimic, bulkier\"),\n    \"2,6-difluorophenol\":     dict(pKa=7.1, logP=1.9, MW=130, HBD=1, HBA=1, note=\"lipophilic acid mimic\"),\n    \"hydroxamic acid\":        dict(pKa=9.0, logP=-0.4, MW=61, HBD=2, HBA=3, note=\"weaker acid, metal binder\"),\n    \"primary amide (CONH2)\":  dict(pKa=None, logP=-1.5, MW=44, HBD=2, HBA=2, note=\"neutral\"),\n    \"methyl ester\":           dict(pKa=None, logP=0.6, MW=59, HBD=0, HBA=2, note=\"neutral, hydrolysable\"),\n}\n\ndef bioisostere_swap(replacement=\"tetrazole\", need_acidic=True, need_permeable=True):\n    p = BIOISOSTERE[\"carboxylic acid (COOH)\"]; r = BIOISOSTERE[replacement]\n    keeps_acid = r[\"pKa\"] is not None and r[\"pKa\"] < 6.5\n    more_lipo = r[\"logP\"] > p[\"logP\"]\n    verdict = []\n    if need_acidic:\n        verdict.append((\"keeps the ionised acid at pH 7.4\", keeps_acid))\n    if need_permeable:\n        verdict.append((\"raises lipophilicity for permeability\", more_lipo))\n    verdict.append((\"molecular weight increase under 60 Da\", r[\"MW\"] - p[\"MW\"] < 60))\n    f, ax = _fig(8.8, 3.6)\n    labels = [\"pKa\", \"logP\", \"MW (Da)\", \"HBD\", \"HBA\"]\n    par = [p[\"pKa\"] if p[\"pKa\"] else 0, p[\"logP\"], p[\"MW\"] / 20, p[\"HBD\"], p[\"HBA\"]]\n    rep = [r[\"pKa\"] if r[\"pKa\"] else 0, r[\"logP\"], r[\"MW\"] / 20, r[\"HBD\"], r[\"HBA\"]]\n    i = np.arange(len(labels)); w = .38\n    ax.bar(i - w / 2, par, w, label=\"carboxylic acid (parent)\", color=MUTED)\n    ax.bar(i + w / 2, rep, w, label=replacement, color=PRIMARY)\n    ax.set_xticks(i); ax.set_xticklabels(labels)\n    ax.set_title(f\"{replacement} - {r['note']}   (MW shown /20)\")\n    ax.legend(fontsize=8); ax.axhline(0, color=\"k\", lw=.8)\n    plt.tight_layout(); plt.show()\n    for text, ok in verdict:\n        print((\"  MET      \" if ok else \"  NOT MET  \") + text)\n    return dict(replacement=replacement, pKa=r[\"pKa\"], logP=r[\"logP\"],\n                dMW=r[\"MW\"] - p[\"MW\"], requirements_met=sum(1 for _, o in verdict if o),\n                requirements_total=len(verdict))\n\n\n# ---------------------------------------------------------------- L32\ndef prodrug_kinetics(promoiety=\"ethyl ester\", esterase_activity=1.0, hours=8.0):\n    \"\"\"First-order activation: prodrug -> active drug, and patient variability.\"\"\"\n    K = {\"ethyl ester\": 0.45, \"5-indanyl ester (EWG)\": 1.60,\n         \"pivaloyl ester (hindered)\": 0.10, \"amide\": 0.02, \"no promoiety\": 0.0}\n    k = K[promoiety] * max(esterase_activity, 0.0)\n    t = np.linspace(0, hours, 400)\n    pro = np.exp(-k * t); act = 1 - pro\n    f, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.3))\n    a1.plot(t, pro * 100, color=MUTED, lw=2, label=\"prodrug (inactive)\")\n    a1.plot(t, act * 100, color=PRIMARY, lw=2.4, label=\"active drug\")\n    a1.set_xlabel(\"hours\"); a1.set_ylabel(\"% of dose\"); a1.legend(fontsize=8)\n    a1.set_title(f\"{promoiety}   k = {k:.2f} /h\")\n    for lab, mult, col in [(\"low enzyme (x0.3)\", .3, SECOND),\n                           (\"typical (x1)\", 1., PRIMARY),\n                           (\"high enzyme (x3)\", 3., \"#15803D\")]:\n        a2.plot(t, (1 - np.exp(-K[promoiety] * mult * t)) * 100, lw=2.2,\n                color=col, label=lab)\n    a2.set_xlabel(\"hours\"); a2.set_ylabel(\"% converted to active drug\")\n    a2.set_title(\"same prodrug, three patients\"); a2.legend(fontsize=8)\n    for ax in (a1, a2):\n        ax.grid(alpha=.25); ax.set_ylim(-3, 103)\n        for s in (\"top\", \"right\"): ax.spines[s].set_visible(False)\n    plt.tight_layout(); plt.show()\n    half = float(np.log(2) / k) if k > 0 else float(\"inf\")\n    at2 = float((1 - np.exp(-k * 2)) * 100)\n    spread = float(abs((1 - np.exp(-K[promoiety] * 3 * 2)) -\n                       (1 - np.exp(-K[promoiety] * .3 * 2))) * 100)\n    return dict(promoiety=promoiety, k_per_h=round(k, 3),\n                half_life_h=(round(half, 2) if np.isfinite(half) else None),\n                pct_active_at_2h=round(at2, 1),\n                patient_spread_at_2h_pct=round(spread, 1))\n\n\n# ---------------------------------------------------------------- L22 extra\ndef method_choice(atoms=20, want=\"partial charges and an electrostatic map\"):\n    \"\"\"Which level of theory answers the question at an affordable cost.\"\"\"\n    needs_electrons = want in (\"partial charges and an electrostatic map\",\n                               \"heat of formation\",\n                               \"transition state of a reaction\")\n    if not needs_electrons:\n        rec = \"Molecular mechanics\"\n        why = (\"The question is about geometry and sterics only. A force field \"\n               \"treats atoms as spheres on springs and never computes an \"\n               \"electron distribution, which is all the speed comes from.\")\n    elif atoms <= 30:\n        rec = \"Ab initio quantum mechanics\"\n        why = (\"Electron distribution is required and the molecule is small \"\n               \"enough that a first-principles calculation is affordable.\")\n    elif atoms <= 150:\n        rec = \"Semi-empirical quantum mechanics\"\n        why = (\"Electron distribution is required but the molecule is too large \"\n               \"for ab initio, so experimentally fitted parameters are used to \"\n               \"simplify the integrals.\")\n    else:\n        rec = \"Molecular mechanics, and reconsider the question\"\n        why = (\"No quantum method is affordable at this size. Either cut the \"\n               \"molecule down to the fragment that matters, or accept that \"\n               \"electronic properties are out of reach.\")\n    cost = {\"Molecular mechanics\": 1, \"Semi-empirical quantum mechanics\": 1e3,\n            \"Ab initio quantum mechanics\": 1e6}\n    f, ax = _fig(8.6, 2.9)\n    names = list(cost); vals = [cost[n] for n in names]\n    cols = [PRIMARY if n in rec else MUTED for n in names]\n    ax.barh(names, vals, color=cols)\n    ax.set_xscale(\"log\"); ax.set_xlabel(\"relative cost (log scale)\")\n    ax.set_title(f\"{atoms} atoms   -   {want}\")\n    plt.tight_layout(); plt.show()\n    print(\"RECOMMEND:\", rec); print()\n    print(_tw(why))\n    return dict(atoms=atoms, question=want, recommend=rec,\n                needs_electron_distribution=needs_electrons)\n\n\n# ---------------------------------------------------------------- L25 extra\ndef site_surface(probe_radius=1.4, show_probe=True):\n    \"\"\"van der Waals surface versus the solvent-accessible (Connolly) surface.\n\n    Two gate atoms are placed so the opening between their van der Waals\n    surfaces is exactly GAP angstroms. A spherical probe passes only if its\n    radius does not exceed GAP/2, so the narration below is derived from the\n    coordinates rather than asserted.\n    \"\"\"\n    R = 1.7\n    GAP = 1.60                       # surface-to-surface opening, angstroms\n    d = 2 * R + GAP                  # required centre-to-centre separation\n    gate = [(-d / 2, 0.0, R), (d / 2, 0.0, R)]\n    wall = [(-5.6, -1.0, R), (-5.0, -3.0, R), (-3.4, -4.4, R),\n            (-1.2, -5.0, R), (1.2, -5.0, R), (3.4, -4.4, R),\n            (5.0, -3.0, R), (5.6, -1.0, R)]\n    atoms = gate + wall\n    half_gap = GAP / 2\n    fits = probe_radius <= half_gap\n\n    gx, gy = np.meshgrid(np.linspace(-8, 8, 460), np.linspace(-7.5, 4.5, 360))\n    vdw = np.zeros_like(gx, bool); excluded = np.zeros_like(gx, bool)\n    for (cx, cy, r) in atoms:\n        dd = (gx - cx) ** 2 + (gy - cy) ** 2\n        vdw |= dd <= r ** 2\n        excluded |= dd <= (r + probe_radius) ** 2\n\n    f, ax = _fig(7.4, 6.0)\n    ax.contourf(gx, gy, excluded.astype(float), levels=[.5, 1.5],\n                colors=[\"#CBD5E1\"], alpha=.8)\n    ax.contour(gx, gy, excluded.astype(float), levels=[.5], colors=[SECOND],\n               linewidths=2.2)\n    ax.contourf(gx, gy, vdw.astype(float), levels=[.5, 1.5], colors=[MUTED],\n                alpha=.95)\n    ax.contour(gx, gy, vdw.astype(float), levels=[.5], colors=[\"#334155\"],\n               linewidths=1.3)\n    if show_probe:\n        ax.add_patch(plt.Circle((0.0, 0.0), probe_radius, fill=False,\n                                ec=PRIMARY, lw=2.4, ls=\"--\"))\n        ax.plot([0], [0], \"x\", color=PRIMARY, ms=9)\n    ax.annotate(\"\", xy=(-d / 2 + R, 0), xytext=(d / 2 - R, 0),\n                arrowprops=dict(arrowstyle=\"<->\", color=PRIMARY, lw=1.6))\n    ax.text(0, .45, f\"{GAP:.2f} A opening\", ha=\"center\", color=PRIMARY,\n            fontsize=9)\n    ax.set_aspect(\"equal\"); ax.set_xlim(-8, 8); ax.set_ylim(-7.5, 4.5)\n    ax.set_xticks([]); ax.set_yticks([])\n    ax.set_title(f\"probe radius {probe_radius:.2f} A\\n\"\n                 \"dark grey = atoms (van der Waals)    \"\n                 \"orange outline = surface this probe can reach\")\n    plt.tight_layout(); plt.show()\n\n    print(f\"opening between the gate atoms : {GAP:.2f} A\")\n    print(f\"largest probe that passes      : {half_gap:.2f} A radius\")\n    print(f\"this probe                     : {probe_radius:.2f} A radius\")\n    print()\n    print(_tw(\n        (\"The probe fits through the opening, so the cavity behind the gate is \"\n         \"mapped as accessible and a ligand atom may be placed there.\"\n         if fits else\n         \"The probe is too large to pass, so the opening is sealed over. The \"\n         \"cavity behind it never appears on the docking surface, which is the \"\n         \"point: no ligand atom could reach it either.\")))\n    return dict(probe_radius=probe_radius, opening_angstrom=GAP,\n                max_probe_radius_that_fits=half_gap,\n                probe_passes=bool(fits),\n                excluded_volume_grid_points=int(excluded.sum()))\n\n\n# ------------------------------------------------- L31, real SwissBioisostere\ndef bioisostere_real(parent=\"carboxylic acid\", replacement=\"primary amide\",\n                     need_not_worse=0.80, max_mw_increase=60.0, show=True):\n    \"\"\"Judge a replacement against real matched-pair data.\n\n    Data: SwissBioisostere (CC-BY 4.0), ChEMBL 36. See swissbio.py for the\n    full citation and for what was changed on retrieval.\n    \"\"\"\n    import swissbio as sb\n    rows = {r[\"name\"]: r for r in sb.table(parent)}\n    if replacement not in rows:\n        print(f\"'{replacement}' is not in the curated {parent} list.\")\n        print(\"Available:\", \", \".join(sorted(rows)[:12]), \"...\")\n        return None\n    r = rows[replacement]\n    n = r[\"better\"] + r[\"similar\"] + r[\"worse\"]\n    not_worse = (r[\"better\"] + r[\"similar\"]) / n\n    total = sb.QUERIES[parent][\"total\"]\n\n    if show:\n        f, (a1, a2) = plt.subplots(1, 2, figsize=(12.6, 4.2))\n        # how activity changed across the real pairs\n        a1.barh([\"worse\", \"similar\", \"better\"],\n                [r[\"worse\"], r[\"similar\"], r[\"better\"]],\n                color=[SECOND, MUTED, \"#15803D\"])\n        a1.set_xlabel(f\"matched molecular pairs (n = {n})\")\n        a1.set_title(f\"{replacement}\\nreplacing {parent} in real compounds\")\n        for s in (\"top\", \"right\"): a1.spines[s].set_visible(False)\n\n        # where this replacement sits against the others by frequency\n        tab = sb.table(parent)\n        ranks = [x[\"rank\"] for x in tab]; freqs = [x[\"freq\"] for x in tab]\n        cols = [PRIMARY if x[\"name\"] == replacement else \"#CBD5E1\" for x in tab]\n        a2.bar(ranks, freqs, color=cols)\n        a2.set_xlabel(f\"rank by frequency (top {len(tab)} of {total:,})\")\n        a2.set_ylabel(\"matched pairs\")\n        a2.set_title(f\"this replacement is rank {r['rank']} of {total:,}\")\n        for s in (\"top\", \"right\"): a2.spines[s].set_visible(False)\n        plt.tight_layout(); plt.show()\n\n    print(f\"SMILES              {r['smiles']}\")\n    print(f\"frequency           {r['freq']:,} matched pairs  (rank {r['rank']} of {total:,})\")\n    print(f\"activity outcome    {r['better']} better / {r['similar']} similar / {r['worse']} worse\")\n    print(f\"not worse           {not_worse*100:.0f}% of pairs\")\n    print(f\"mean change         logP {r['dlogp']:+.2f}   tPSA {r['dtpsa']:+.1f}   MW {r['dmw']:+.1f}\")\n    print()\n    checks = [(f\"at least {need_not_worse*100:.0f}% of pairs not worse\", not_worse >= need_not_worse),\n              (f\"molecular weight increase under {max_mw_increase:.0f} Da\",\n               r[\"dmw\"] < max_mw_increase)]\n    for text, okd in checks:\n        print((\"  MET      \" if okd else \"  NOT MET  \") + text)\n    return dict(name=replacement, smiles=r[\"smiles\"], freq=r[\"freq\"], rank=r[\"rank\"],\n                not_worse=round(not_worse, 3), dlogp=r[\"dlogp\"], dtpsa=r[\"dtpsa\"],\n                dmw=r[\"dmw\"], met=sum(1 for _, o in checks if o), total_checks=len(checks))\n\n\ndef bioisostere_ranking(parent=\"carboxylic acid\", min_freq=300, show=True):\n    \"\"\"Frequency is not success: rank the common replacements by outcome.\"\"\"\n    import swissbio as sb\n    tab = [r for r in sb.table(parent) if r[\"freq\"] >= min_freq]\n    for r in tab:\n        n = r[\"better\"] + r[\"similar\"] + r[\"worse\"]\n        r[\"not_worse\"] = (r[\"better\"] + r[\"similar\"]) / n if n else 0\n    by_freq = sorted(tab, key=lambda r: -r[\"freq\"])\n    by_ok = sorted(tab, key=lambda r: -r[\"not_worse\"])\n    if show:\n        f, ax = _fig(9.6, 5.0)\n        ax.scatter([r[\"freq\"] for r in tab], [r[\"not_worse\"] * 100 for r in tab],\n                   s=70, color=PRIMARY, zorder=3)\n        for r in tab:\n            if r[\"freq\"] > 600 or r[\"not_worse\"] > 0.88:\n                ax.annotate(r[\"name\"], (r[\"freq\"], r[\"not_worse\"] * 100),\n                            fontsize=8, xytext=(5, 4), textcoords=\"offset points\")\n        ax.set_xscale(\"log\")\n        ax.set_xlabel(f\"how often it was done (matched pairs, log scale)\")\n        ax.set_ylabel(\"% of pairs where activity was not worse\")\n        ax.set_title(f\"replacing {parent}: popularity against outcome\")\n        plt.tight_layout(); plt.show()\n    print(f\"{'most common':<30}{'most reliable':<30}\")\n    print(\"-\" * 60)\n    for i in range(min(6, len(tab))):\n        print(f\"{by_freq[i]['name'][:28]:<30}{by_ok[i]['name'][:28]:<30}\")\n    return dict(n=len(tab),\n                most_common=by_freq[0][\"name\"], most_reliable=by_ok[0][\"name\"],\n                same=by_freq[0][\"name\"] == by_ok[0][\"name\"])\n", "sims.py", "exec"), _sm.__dict__)
sys.modules["sims"] = _sm
sims = _sm

ITEMS = [
    Item("P1", "What does virtual screening of a compound database deliver?",
         {"A": "A measured binding affinity for each compound in the database", "B": "A ranked shortlist of compounds worth testing experimentally", "C": "A synthetic route to every compound the ranking has selected", "D": "A guarantee that the highest ranked compound will prove active"},
         "B", {"C": "Synthetic routes are a separate chemistry problem entirely.", "D": "No ranking carries a guarantee; that is why compounds are then tested.", "A": "Docking scores are estimates, not measured affinities.", "_correct": "Virtual screening prioritises a database so that limited experimental capacity is spent on the most promising compounds."},
         bloom="Understand", silo="", section="precheck"),
    Item("P2", "A docking protocol reproduces a known crystal pose to 1.2 angstrom. What has been validated?",
         {"A": "That the scoring function ranks diverse compounds correctly", "B": "That the protein structure used is free of experimental error", "C": "That any compound this protocol ranks highly will be active", "D": "That the search can find the right geometry for this ligand"},
         "D", {"A": "Reproducing one pose says nothing about ranking across compounds.", "C": "Correct posing does not establish that scoring and ranking are reliable.", "B": "The crystal structure's own errors are untouched by a redocking test.", "_correct": "Redocking tests pose prediction only: the search found the right geometry, which leaves ranking entirely unproven."},
         bloom="Understand", silo="", section="precheck"),
    Item("P3", "Why is a homology model below about 40 percent sequence identity treated with caution?",
         {"A": "Because the model cannot be energy minimized at that identity", "B": "Because the protein fold is certain to be completely different", "C": "Because errors in the binding site itself become common there", "D": "Because docking programs refuse to read homology model files"},
         "C", {"A": "Minimization runs perfectly well on a model of any quality.", "D": "Docking programs read the coordinates without objection, which is the danger.", "B": "The fold is often broadly right; it is the detail that becomes unreliable.", "_correct": "Below roughly 40 percent identity, side chains and loops in the site are frequently misplaced, so poses and scores rest on a misbuilt cavity."},
         bloom="Understand", silo="", section="precheck"),
    Item("C1", "At a narrow search width of 2 angstrom 32 of the 220 poses fall within 2 angstrom of the crystal pose and the best-scoring one has an RMSD of about 0.45 angstrom. What does this result establish about scoring functions?",
         {"A": "It is an artefact of starting near the answer and so teaches nothing", "B": "Scoring functions are reliable in general, so pose prediction is solved", "C": "Near the correct answer the score is informative, so local search works", "D": "The score must be overfitted to this complex, since it succeeded here"},
         "C", {"B": "The next cell widens the search and the same score begins ranking wrong poses first.", "D": "This score has no fitted parameters tied to this complex; it is contact, hydrogen bonding and overlap.", "A": "That local reliability is precisely what makes refinement and local search effective; it is a real property.", "_correct": "Scoring functions tend to be locally informative and globally unreliable. Within a basin around the correct pose the score ranks sensibly, which is why local refinement is effective. The failure comes when the search has to choose between distant basins."},
         bloom="Evaluate", silo="6", section="concept"),
    Item("C2", "Widening the search to 5 angstrom, the ranked list shows a pose at 4.79 angstrom RMSD scoring -32 while a pose at 1.71 angstrom scores +2. How should that inversion change how you read a docking result?",
         {"A": "The scoring function should be discarded, as it ranked a wrong pose above a correct one", "B": "The top-ranked pose cannot be trusted on its score alone and needs independent checking", "C": "The correct pose must have been generated badly, since a correct pose should always score best", "D": "Nothing changes, since the top pose overall was still approximately correct"},
         "B", {"C": "Both poses are legitimate rigid placements of the real ligand; neither was generated badly.", "D": "An inversion anywhere in the top ranks means rank order is not a reliable guide to correctness.", "A": "The same score is informative near the answer; discarding it loses that. The lesson is about its range of validity.", "_correct": "This is the central practical limitation of docking. A score can rank a wrong pose above a correct one, so a docking result is a hypothesis to be tested against known structure-activity data, mutagenesis or a crystal structure, not an answer to be accepted on rank."},
         bloom="Evaluate", silo="6", section="concept"),
    Item("C3", "Re-running the calculation with eight different random seeds, the score picked a correct pose in seven of them and failed in one. What is the right way to report a docking protocol's performance?",
         {"A": "From the first run, since changing the seed is a form of result selection", "B": "From the single best run, since that shows what the method can achieve", "C": "Over many independent runs, since a single run can succeed or fail by chance", "D": "Performance cannot be reported, because stochastic methods are not reproducible"},
         "C", {"B": "Reporting the best run overstates performance and is not reproducible by anyone else.", "D": "Stochastic methods are reproducible in distribution, which is why seeds are recorded and multiple runs reported.", "A": "Running several seeds and reporting all of them is standard practice, not selection; selection would be reporting only the good ones.", "_correct": "A seven in eight success rate is a useful, honest statement; the single successful run that happens to be first is not. Fixing and reporting seeds lets others reproduce your work, and reporting the distribution rather than the best case is what makes a benchmark meaningful."},
         bloom="Evaluate", silo="6", section="concept"),
    Item("Q1", "Redocking a co-crystallized ligand reproduces the crystal pose to 1.2 A, but the same protocol ranks a known nanomolar inhibitor 400th out of 1,000 compounds. What does this pattern indicate?",
         {"A": "The protein structure used for docking must be incorrectly prepared", "B": "Pose prediction is working while the scoring function ranks poorly", "C": "The known inhibitor probably binds at an entirely separate site", "D": "The redocking result must be a coincidence and should be repeated"},
         "B", {"A": "is contradicted by the successful redock", "C": "is speculative", "D": "dismisses a valid validation result", "_correct": "Reproducing the crystal pose shows the search and geometry work, while misranking a known nanomolar inhibitor is a scoring-function failure."},
         bloom="Evaluate", silo="6", section="post"),
    Item("Q2", "A docking campaign must process 80,000 compounds without an operator selecting an anchor for each. Which program design fits that constraint?",
         {"A": "Hammerhead, because fragments are scored and heads chosen by program", "B": "FlexX, because chemical complementarity places the anchor precisely", "C": "Manual docking, because an expert resolves ambiguity most reliably", "D": "FLOG, because stored conformer libraries remove the anchor problem"},
         "A", {"C": "is infeasible at scale", "D": "addresses flexibility, not anchoring", "B": "names the program whose anchor is chosen manually, the stated obstacle", "_correct": "Hammerhead scores fragments and selects head fragments by program, so no operator input is needed for each of the 80,000 compounds."},
         bloom="Apply", silo="6", section="post"),
    Item("Q3", "A target has 34 percent sequence identity to the nearest available crystal structure. A colleague proposes docking a library into a homology model built from it. What is the soundest response?",
         {"A": "Treat results cautiously, as serious errors are common below 40 percent", "B": "Proceed, because identity affects loops but not the binding site", "C": "Proceed, since any homology model outperforms no structural model", "D": "Abandon structure-based work and use ligand-based methods only"},
         "A", {"B": "Below 40 percent identity the binding site itself is often misbuilt, not only the loops", "D": "Abandoning structure-based work overcorrects, since cautious use or a pseudoreceptor remains", "C": "A model below 40 percent identity can be worse than none if it misleads the design", "_correct": "Below roughly 40 percent identity, homology models frequently misplace side chains and backbone within the binding site itself."},
         bloom="Evaluate", silo="6", section="post"),
    Item("Q4", "Two docked poses of one ligand differ by a 180 degree flip and score within 0.2 kcal/mol of each other. What is the appropriate conclusion?",
         {"A": "The docking run failed and the site definition must be rebuilt", "B": "The higher-scoring pose should be adopted as the binding mode", "C": "The ligand is likely to occupy both orientations simultaneously", "D": "The scoring function cannot distinguish them, so seek experimental data"},
         "D", {"B": "over-trusts a difference inside the method's error", "C": "misreads degeneracy as physical co-occupancy", "A": "is unjustified", "_correct": "A 0.2 kcal/mol gap lies well inside the error of any scoring function, so the two poses cannot be separated computationally."},
         bloom="Evaluate", silo="6", section="post"),
    Item("Q5", "In an anchor-and-grow run, the fragment scoring highest as an anchor yields a poorer final complex than a lower-scoring fragment does. Why is this expected?",
         {"A": "An anchor's own score ignores how the rest of the ligand will fit", "B": "Energy minimization after each addition erases the anchor's lead", "C": "Torsion angles are varied systematically, which biases the result", "D": "Growth is pruned after each layer, which discards good constructs"},
         "A", {"C": "describes normal procedure", "B": "misattributes the effect to minimization", "D": "describes pruning, not the cause", "_correct": "An anchor is scored on its own interactions, which say nothing about whether the remaining fragments can be grown productively into the site."},
         bloom="Analyze", silo="6", section="post"),
]

gb = Gradebook("L26", "Molecular Docking Studies II")
gb.add_many(ITEMS)
print("Ready - 11 questions loaded. Scroll down and begin.")

## Learning objectives

| # | By the end of this activity you should be able to | Bloom | Syllabus level | SILO |
|---|---|---|---|---|
| 1 | Critique docked poses, identifying two reasons a top score can mislead. | Evaluate | II | 3 |
| 2 | Explain how virtual screening of databases reduces experimental workload. | Understand | V | 6 |
| 3 | Interpret a hit rate and an enrichment factor from a screening result. | Apply | IV | 6 |
| 4 | State when a homology model or a pseudoreceptor can stand in for an X-ray structure. | Apply | IV | 6 |

*Syllabus levels follow the PHS 312 legend, in which Level I is Creating and Level VI is Remembering.*

## Part 1 - What do you already know?

Three questions before you begin. **You will not be told whether you are
right**, and that is deliberate: the point is to measure what you knew coming
in, so that at the end you can see how much the activity moved you. Answer
from what you remember from the reading. Guessing is fine.

In [ ]:
#@title P1   (run after choosing) { display-mode: "form" }
#@markdown **P1.**  What does virtual screening of a compound database deliver?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. A measured binding affinity for each compound in the database", "B. A ranked shortlist of compounds worth testing experimentally", "C. A synthetic route to every compound the ranking has selected", "D. A guarantee that the highest ranked compound will prove active"]
gb.answer("P1", response[:1])

In [ ]:
#@title P2   (run after choosing) { display-mode: "form" }
#@markdown **P2.**  A docking protocol reproduces a known crystal pose to 1.2 angstrom. What has been validated?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. That the scoring function ranks diverse compounds correctly", "B. That the protein structure used is free of experimental error", "C. That any compound this protocol ranks highly will be active", "D. That the search can find the right geometry for this ligand"]
gb.answer("P2", response[:1])

In [ ]:
#@title P3   (run after choosing) { display-mode: "form" }
#@markdown **P3.**  Why is a homology model below about 40 percent sequence identity treated with caution?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Because the model cannot be energy minimized at that identity", "B. Because the protein fold is certain to be completely different", "C. Because errors in the binding site itself become common there", "D. Because docking programs refuse to read homology model files"]
gb.answer("P3", response[:1])

## Part 2 - Does the score pick the right pose

This is the question that decides whether docking is useful. The
cell generates many real poses of tacrine by displacing and rotating the
crystallographic ligand, scores each one, and plots **score against RMSD to
the crystal pose**.

> **About this structure.** 1ACJ is *Torpedo californica*
> acetylcholinesterase with tacrine bound, from the Protein Data Bank
> (Harel *et al.*, *Proc. Natl. Acad. Sci. USA* 1993, **90**, 9031).
> The coordinates are embedded in this notebook, so the cell below
> downloads nothing and sends nothing anywhere. No personal data of any
> kind is involved, and nothing you type is transmitted or stored.

If the score were perfect, the plot would slope cleanly: better score,
lower RMSD. Run it at a narrow search width first, then widen it.

In [ ]:
#@title Score against crystallographic truth { display-mode: "form" }
#@markdown How far from the crystal pose should the search wander?
search_width_angstrom = 2.0 #@param {type:"slider", min:1.0, max:6.0, step:0.5}
number_of_poses = 220 #@param {type:"slider", min:60, max:400, step:20}
r = crystal.score_vs_truth(n=number_of_poses, spread=search_width_angstrom)

In [ ]:
#@title C1   (run after choosing) { display-mode: "form" }
#@markdown **C1.**  At a narrow search width of 2 angstrom 32 of the 220 poses fall within 2 angstrom of the crystal pose and the best-scoring one has an RMSD of about 0.45 angstrom. What does this result establish about scoring functions?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Near the correct answer the score is informative, so local search works", "B. Scoring functions are reliable in general, so pose prediction is solved", "C. It is an artefact of starting near the answer and so teaches nothing", "D. The score must be overfitted to this complex, since it succeeded here"]
gb.answer("C1", response[:1])

In [ ]:
#@title C2   (run after choosing) { display-mode: "form" }
#@markdown **C2.**  Widening the search to 5 angstrom, the ranked list shows a pose at 4.79 angstrom RMSD scoring -32 while a pose at 1.71 angstrom scores +2. How should that inversion change how you read a docking result?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. The top-ranked pose cannot be trusted on its score alone and needs independent checking", "B. The scoring function should be discarded, as it ranked a wrong pose above a correct one", "C. The correct pose must have been generated badly, since a correct pose should always score best", "D. Nothing changes, since the top pose overall was still approximately correct"]
gb.answer("C2", response[:1])

In [ ]:
#@title C3   (run after choosing) { display-mode: "form" }
#@markdown **C3.**  Re-running the calculation with eight different random seeds, the score picked a correct pose in seven of them and failed in one. What is the right way to report a docking protocol's performance?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Over many independent runs, since a single run can succeed or fail by chance", "B. From the single best run, since that shows what the method can achieve", "C. From the first run, since changing the seed is a form of result selection", "D. Performance cannot be reported, because stochastic methods are not reproducible"]
gb.answer("C3", response[:1])

## Part 3 - End-of-activity assessment

Five questions in the style of the course quizzes and exams, mapped to the
Student Intended Learning Outcomes for this unit (**SILO 6**).
This time you get full feedback, including why each wrong option is wrong.
Answer each one, then run the summary cell at the bottom.

In [ ]:
#@title Q1   (run after choosing) { display-mode: "form" }
#@markdown **Q1.**  Redocking a co-crystallized ligand reproduces the crystal pose to 1.2 A, but the same protocol ranks a known nanomolar inhibitor 400th out of 1,000 compounds. What does this pattern indicate?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. The protein structure used for docking must be incorrectly prepared", "B. Pose prediction is working while the scoring function ranks poorly", "C. The known inhibitor probably binds at an entirely separate site", "D. The redocking result must be a coincidence and should be repeated"]
gb.answer("Q1", response[:1])

In [ ]:
#@title Q2   (run after choosing) { display-mode: "form" }
#@markdown **Q2.**  A docking campaign must process 80,000 compounds without an operator selecting an anchor for each. Which program design fits that constraint?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Hammerhead, because fragments are scored and heads chosen by program", "B. FlexX, because chemical complementarity places the anchor precisely", "C. Manual docking, because an expert resolves ambiguity most reliably", "D. FLOG, because stored conformer libraries remove the anchor problem"]
gb.answer("Q2", response[:1])

In [ ]:
#@title Q3   (run after choosing) { display-mode: "form" }
#@markdown **Q3.**  A target has 34 percent sequence identity to the nearest available crystal structure. A colleague proposes docking a library into a homology model built from it. What is the soundest response?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Treat results cautiously, as serious errors are common below 40 percent", "B. Proceed, because identity affects loops but not the binding site", "C. Proceed, since any homology model outperforms no structural model", "D. Abandon structure-based work and use ligand-based methods only"]
gb.answer("Q3", response[:1])

In [ ]:
#@title Q4   (run after choosing) { display-mode: "form" }
#@markdown **Q4.**  Two docked poses of one ligand differ by a 180 degree flip and score within 0.2 kcal/mol of each other. What is the appropriate conclusion?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. The docking run failed and the site definition must be rebuilt", "B. The higher-scoring pose should be adopted as the binding mode", "C. The ligand is likely to occupy both orientations simultaneously", "D. The scoring function cannot distinguish them, so seek experimental data"]
gb.answer("Q4", response[:1])

In [ ]:
#@title Q5   (run after choosing) { display-mode: "form" }
#@markdown **Q5.**  In an anchor-and-grow run, the fragment scoring highest as an anchor yields a poorer final complex than a lower-scoring fragment does. Why is this expected?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. An anchor's own score ignores how the rest of the ligand will fit", "B. Energy minimization after each addition erases the anchor's lead", "C. Torsion angles are varied systematically, which biases the result", "D. Growth is pruned after each layer, which discards good constructs"]
gb.answer("Q5", response[:1])

## Your results

In [ ]:
#@title Show my performance and my report code { display-mode: "form" }
gb.summary()
print()
_ = gb.report_code()

### What to submit

1. Copy the single `L26|...` line printed above.
2. Paste it into the Canvas assignment **L26 Colab activity**.
3. With your partner, be ready to say one thing the simulation showed you that
   the reading did not.

Your report code records how many questions you answered correctly. It is a
participation and self-check record, not a proctored exam score.